[ja]: #
# Chapter 7. 情報処理容量

[en]: #
# Chapter 7. Information Processing Capacity

[zh]: #
# 第 7 章 信息处理容量

[ja]: #
この章では、前章の記憶容量を拡張した指標である情報処理容量(Information Processing Capacity; IPC)の計算方法を学習します。
また前章同様、階数やリアプノフ指数などの力学系の性質と、得られる情報処理容量との間の関係を学びましょう。

**注意:** この章の後半部では、GPUを使用した環境が推奨されます。
手元のPCにGPUがない場合はGoogle Colaboratory上での実行をお勧めします。

[en]: #
In this chapter, we will learn how to calculate the information processing capacity (IPC), an extended metric of the memory capacity introduced in the previous chapter.
As in the previous chapter, we will also explore the relationship between the properties of dynamical systems, such as rank and dynamics, and the resulting IPC.

**Note:** In the latter part of this chapter, an environment with GPU support is recommended.
If your local PC does not have a GPU, it is recommended to run it on Google Colaboratory.

[zh]: #
在本章中，我们将学习如何计算信息处理容量，这是上一章中介绍的记忆容量的扩展度量。
与上一章一样，我们还将探讨动力系统的属性(例如等级和动态)与生成的 IPC 之间的关系。

**注：**本章后半部分推荐使用支持GPU的环境。
如果您的本地PC没有GPU，建议在Google Colaboratory上运行。

[ja]: #
## 前書き

[en]: #
## Introduction

[zh]: #
## 介绍

[ja]: #
情報処理容量はJ. Dambreら<sup>[1]</sup>が提案した記憶容量を拡張した指標で、入力時系列に対してどのような計算 (記憶と非線形性) を力学系が行っているかを評価します。
前章と同じく以下の式で表される $1$ 入力 $N$ 次元の力学系 $x[k]$ とある線形写像 $g: \mathbb{R}^N \to \mathbb{R}$ による出力 $\hat{y}[k]$ を考えます。

[en]: #
IPC is an extended metric of memory capacity proposed by J. Dambre et al.<sup>[1]</sup>, which evaluates what kind of computations (memory and nonlinearity) a dynamical system performs on input time series.
As in the previous chapter, we consider a single-input $N$-dimensional dynamical system $x[k]$ represented by the following equations and the output $\hat{y}[k]$ obtained through a certain linear mapping $g: \mathbb{R}^N \to \mathbb{R}$:

[zh]: #
IPC是J. Dambre等人<sup>[1]</sup>提出的记忆容量的扩展度量，它评估动力系统对输入时间序列执行什么样的计算(记忆和非线性)。
与上一章一样，我们考虑由以下方程表示的单输入 $N$ 维动力系统 $x[k]$ 以及通过一定的线性映射 $g: \mathbb{R}^N \to \mathbb{R}$ 得到的输出 $\hat{y}[k]$：

[END]: #
$$
\renewcommand{\Tau}{\mathrm{T}}
\renewcommand{\Zeta}{\mathrm{Z}}
\begin{align*}
x[k+1] &= f \left(x[k],\zeta[k+1]\right) \\
\hat{y}[k] &= g(x[k])
,\end{align*}
$$

[ja]: #
また入力時系列 $\zeta[k]$ は零平均で定常的かつi.i.d.であると仮定します。
ここで新たに 以下の式で定義される容量 $\mathrm{C}$ を導入します。

[en]: #
where the input time series $\zeta[k]$ is assumed to be zero-mean, stationary, and i.i.d.
Here, we introduce a new capacity $\mathrm{C}$ defined by the following equation:

[zh]: #
其中输入时间序列 $\zeta[k]$ 假设为零均值、平稳且独立同分布。
在这里，我们引入一个新容量 $\mathrm{C}$，其定义如下：

[END]: #
$$
\begin{align*}
\mathrm{C}[x, z] := \mathrm{R}^2[z, x]
.\end{align*}
$$

[ja]: #
式に示されているとおり$\mathrm{C}(x, z)$ は目標時系列 $z$ をどれほど $x$ から再構成できるかを定量化した指標で、決定係数 $\mathrm{R}^2$ を用いて計算され、0から1の範囲を取ります。
前章で学習した記憶関数は以下のとおり $C$ を用いて表現されます。

[en]: #
As shown in the equation, $\mathrm{C}(x, z)$ is a metric that quantifies how well the target time series $z$ can be reconstructed from $x$, calculated using the coefficient of determination $\mathrm{R}^2$, and it takes values in the range [0, 1].
The memory function, learned in the previous chapter, is expressed using $\mathrm{C}$ as follows:

[zh]: #
如公式所示，$\mathrm{C}(x, z)$ 是一个度量，用于量化从 $x$ 重建目标时间序列 $z$ 的能力，使用确定系数 $\mathrm{R}^2$ 计算，其取值范围为 [0, 1]。
上一章学到的记忆函数，用$\mathrm{C}$表示如下：

[END]: #
$$
\begin{align*}
\mathrm{MF}[\tau] &= \mathrm{C}[x, \zeta^\tau]
.\end{align*}
$$

[ja]: #
記憶容量$\mathrm{MC}$ は $\tau$ を変えて全過去に対して $\mathrm{C}$ を計算し総和を取って計算されました。
言い換えれば記憶容量は、過去入力 $[\zeta[k], \zeta[k-1], \zeta[k-2],~\ldots]$ の**線形**な変換を、現在の状態 $x[k]$ (の線形変換) からどれほど回収できるかを定量化した指標といえます。
一方で情報処理容量の計算では、**非線形**な範囲まで考慮・拡張してどれほど再構成できるかを評価します。

さて目標として設定する過去入力の非線形な変換 $z$ をどのように構成すれば良いでしょうか？
ここで、目標時系列の直交性と網羅性、すなわち重複なくすべてのパターンを考慮しなければならない点に注意しなければなりません。
なぜならば線形従属な目標を許容すると、総容量はいくらでも大きくできてしまうからです。
記憶容量の計算の際、各記憶関数の値 $\mathrm{C}[x, \zeta^\tau]$ の単純な総和を取って求められたのは、入力の i.i.d.性を仮定しており、$\tau_1 \neq \tau_2$ の時 $\zeta^{\tau_1}$ と $\zeta^{\tau_2}$ が線形独立で直交していたからです ($\mathrm{E}[\zeta^{\tau_1} \zeta^{\tau_2}] = 0$)。

情報処理容量の計算では **直交多項式**<sup>[2]</sup>と呼ばれる道具を用いて、目標時系列を重複なく網羅的に構成します。
直交多項式は**次数**と呼ばれる整数のパラメータを持ちます。
一般に $d$ の値が大きいほど非線形性が強くなります (逆に $d=0$ のときは定数、$d=1$ のときは線形変換に対応)。
$d$ 次の直交多項式を $\mathcal{P}_d$ と表記します。

[en]: #
The memory capacity $\mathrm{MC}$ is calculated by summing $\mathrm{C}$ over all past inputs by varying $\tau$.
In other words, memory capacity can be interpreted as a metric that quantifies how well the **linear** transformation of past inputs $[\zeta[k], \zeta[k-1], \zeta[k-2],~\ldots]$ can be recovered from the current state $x[k]$ (or its linear transformation).
On the other hand, the calculation of IPC evaluates how well reconstruction can be achieved, extending the scope to include **nonlinear** transformations.

Now, how should we construct the nonlinear transformation $z$ of past inputs that we aim to evaluate? The points to consider here are the orthogonality and completeness of the target time series, meaning that all patterns must be considered without duplication.
This is because allowing linearly dependent targets would make the total capacity arbitrarily large.
In the calculation of memory capacity, the simple summation of the values of each memory function $\mathrm{C}[x, \zeta^\tau]$ was valid because the i.i.d. nature of the input was assumed.
That is, for $\tau_1 \neq \tau_2$, $\zeta^{\tau_1}$ and $\zeta^{\tau_2}$ were linearly independent and orthogonal ($\mathrm{E}[\zeta^{\tau_1} \zeta^{\tau_2}] = 0$).

In the calculation of IPC, a tool called **orthogonal polynomials**<sup>[2]</sup> is used to construct target time series comprehensively without duplication.
Orthogonal polynomials have an integer parameter called **degree**.
Generally, the larger the value of $d$, the stronger the nonlinearity (conversely, $d=0$ corresponds to a constant, and $d=1$ corresponds to a linear transformation).
Orthogonal polynomials of degree $d$ are denoted as $\mathcal{P}_d$.

[zh]: #
记忆容量 $\mathrm{MC}$ 是通过改变 $\tau$ 对所有过去输入的 $\mathrm{C}$ 求和来计算的。
换句话说，记忆容量可以解释为一种度量，用于量化从当前状态 $x[k]$(或其线性变换)恢复过去输入 $[\zeta[k], \zeta[k-1], \zeta[k-2],~\ldots]$ 的**线性**变换的程度。
另一方面，IPC 的计算评估重建的效果，将范围扩展到包括**非线性**变换。

现在，我们应该如何构建我们想要评估的过去输入的非线性变换 $z$？这里要考虑的点是目标时间序列的正交性和完整性，这意味着必须考虑所有模式而不重复。
这是因为允许线性相关的目标将使总容量任意大。
在记忆容量的计算中，每个记忆函数 $\mathrm{C}[x, \zeta^\tau]$ 的值的简单求和是有效的，因为输入的 i.i.d. 性质。
也就是说，对于 $\tau_1 \neq \tau_2$，$\zeta^{\tau_1}$ 和 $\zeta^{\tau_2}$ 是线性独立且正交的 ($\mathrm{E}[\zeta^{\tau_1} \zeta^{\tau_2}] = 0$)。

在IPC的计算中，使用了一个名为**正交多项式(Orthogonal polynomial)**<sup>[2]</sup>的工具来全面构建目标时间序列，无需重复。
正交多项式有一个整数参数，称为**度**。
一般情况下，$d$的值越大，非线性越强(反之，$d=0$对应常数，$d=1$对应线性变换)。
$d$ 的正交多项式表示为 $\mathcal{P}_d$。

[ja]: #
まず記憶容量のときに用いられた1次の目標時系列は、$d=1$次の直交多項式 $\mathcal{P}_1$ を用いて改めて以下の形で表記できます。

[en]: #
First, the first-order target time series used in memory capacity can be expressed again in the following form using the first-degree orthogonal polynomial $\mathcal{P}_1$:

[zh]: #
首先，记忆容量中使用的一阶目标时间序列可以使用一阶正交多项式 $\mathcal{P}_1$再次表达为以下形式：

[END]: #
$$
\begin{align*}
\mathcal{P}_1(\zeta^0),~\mathcal{P}_1(\zeta^1),~\mathcal{P}_1(\zeta^2),~\mathcal{P}_1(\zeta^3),~\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^5),~\ldots
.\end{align*}
$$

[ja]: #
次に2次の目標時系列を見ていきましょう。
ここでは2次の直交多項式 $\mathcal{P}_2$ を用いて、以下のように列挙されます。

[en]: #
Next, let us consider second-order target time series.
Here, using the second-degree orthogonal polynomial $\mathcal{P}_2$, they are enumerated as follows:

[zh]: #
接下来，让我们考虑二阶目标时间序列。
这里，使用二次正交多项式 $\mathcal{P}_2$，将它们枚举如下：

[END]: #
$$
\begin{align*}
&\mathcal{P}_2(\zeta^0),~\mathcal{P}_2(\zeta^1),~\mathcal{P}_2(\zeta^2),~\mathcal{P}_2(\zeta^3),~\mathcal{P}_2(\zeta^4),~\mathcal{P}_2(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^1),~\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^2),~\mathcal{P}_{1}(\zeta^0)\mathcal{P}_1(\zeta^3),~\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^2),~\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^3),~\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^3),~\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^3)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^3)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^4)\mathcal{P}_1(\zeta^5),\ldots
.\end{align*}
$$

[ja]: #
ここで2次の目標時系列として、$\mathcal{P}_2$を使った要素だけでなく、$\mathcal{P}_{1}$同士の積 (すなわち$1+1=2$) も含まれる点に注意してください。
直交しているのでいずれの目標時系列も線形独立で、その間の内積は $0$ になります。

3次の目標時系列も同様に列挙されます。
合計次数が3となる足し算のパターンは$3,~2+1, 1+1+1$ の3パターンあるため以下のとおりより考慮されるパターンが増えます。

[en]: #
Here, note that as second-order target time series, not only the elements using $\mathcal{P}_2$ but also the products of $\mathcal{P}_{1}$ (i.e., $1+1=2$) are included.
Since they are orthogonal, all target time series are linearly independent, and the inner product between them is $0$.

Third-order target time series are also enumerated similarly.
Since there are three patterns of addition that result in a total degree of 3: $3,~2+1, 1+1+1$, the number of considered patterns increases as follows:

[zh]: #
这里，请注意，作为二阶目标时间序列，不仅包括使用$\mathcal{P}_2$的元素，还包括$\mathcal{P}_{1}$的乘积(即，$1+1=2$)。
由于它们是正交的，因此所有目标时间序列都是线性独立的，它们之间的内积为 $0$。

三阶目标时间序列也类似地列举。
由于存在三种加法模式，其总次数为 3：$3,~2+1, 1+1+1$，因此考虑的模式数量增加如下：

[END]: #
$$
\begin{align*}
&\mathcal{P}_3(\zeta^0),~\mathcal{P}_3(\zeta^1),~\mathcal{P}_3(\zeta^2),~\mathcal{P}_3(\zeta^3),~\mathcal{P}_3(\zeta^4),~\mathcal{P}_3(\zeta^5),~\ldots\\
&\mathcal{P}_2(\zeta^0)\mathcal{P}_1(\zeta^1),~\mathcal{P}_2(\zeta^0)\mathcal{P}_1(\zeta^2),~\mathcal{P}_2(\zeta^0)\mathcal{P}_1(\zeta^3),~\mathcal{P}_2(\zeta^0)\mathcal{P}_1(\zeta^4),~\mathcal{P}_2(\zeta^0)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_2(\zeta^1)\mathcal{P}_1(\zeta^2),~\mathcal{P}_2(\zeta^1)\mathcal{P}_1(\zeta^3),~\mathcal{P}_2(\zeta^1)\mathcal{P}_1(\zeta^4),~\mathcal{P}_2(\zeta^1)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_2(\zeta^2)\mathcal{P}_1(\zeta^3),~\mathcal{P}_2(\zeta^2)\mathcal{P}_1(\zeta^4),~\mathcal{P}_2(\zeta^2)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_2(\zeta^3)\mathcal{P}_1(\zeta^4),~\mathcal{P}_2(\zeta^3)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_2(\zeta^4)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^2),~\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^3),~\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^3),~\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^3)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^3)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^0)\mathcal{P}_1(\zeta^4)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^3),~\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^3)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^3)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^1)\mathcal{P}_1(\zeta^4)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^3)\mathcal{P}_1(\zeta^4),~\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^3)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^2)\mathcal{P}_1(\zeta^4)\mathcal{P}_1(\zeta^5),~\ldots\\
&\mathcal{P}_1(\zeta^3)\mathcal{P}_1(\zeta^4)\mathcal{P}_1(\zeta^5),~\ldots
.\end{align*}
$$

[ja]: #
$d=4$ 以降も同様かつ爆発的に組み合わせが増えていきます。
これらを一般化し次数を表す整数の集合 $D=\{d_1,d_2,~\ldots,~d_m\}$ と同じ要素数を持つ時間遅れを表す整数の集合 $\Tau=\{\tau_1, \tau_2,~\ldots,~\tau_m\}$ を用いてある目標時系列 $\zeta^{D,\Tau}$ を以下の式で定義します。

[en]: #
The combinations continue to increase explosively for $d = 4$ and beyond as well.
We generalize these and define a target time series $\zeta^{D,\Tau}$ using the following equation, with $D=\{d_1,d_2,~\ldots,~d_m\}$ being a set of integers representing degrees and $\Tau=\{\tau_1, \tau_2,~\ldots,~\tau_m\}$ being a set of integers representing time delays, both having the same number of elements:

[zh]: #
$d = 4$ 及更高版本的组合继续呈爆炸式增长。
我们概括这些并使用以下等式定义目标时间序列 $\zeta^{D,\Tau}$，其中 $D=\{d_1,d_2,~\ldots,~d_m\}$ 是一组表示度数的整数，$\Tau=\{\tau_1, \tau_2,~\ldots,~\tau_m\}$ 是一组表示时间延迟的整数，两者具有相同数量的元素：

[END]: #
$$
\begin{align*}
\zeta^{D,\Tau}[k] :=& \mathcal{P}_{d_1}(\zeta[k-\tau_1])\mathcal{P}_{d_2}(\zeta[k-\tau_2])\cdots \mathcal{P}_{d_m}(\zeta[k-\tau_m]) \\
=& \prod_{i=1}^m \mathcal{P}_{d_i}(\zeta[k-\tau_i])
.\end{align*}
$$

[ja]: #
$d$ 次の情報処理容量 $\mathrm{C}^d$ は $\sum_i d_i = d$ となる $D$ に対する目標時系列の容量の総和で以下の式で定義されます。

[en]: #
The $d$-th order IPC $\mathrm{C}^d$ is defined as the sum of capacities for target time series corresponding to $D$ such that $\sum_i d_i = d$:

[zh]: #
$d$ 阶 IPC $\mathrm{C}^d$ 定义为与 $D$ 对应的目标时间序列的容量总和，使得 $\sum_i d_i = d$：

[END]: #
$$
\begin{align*}
\mathrm{C}^d[x, \zeta] := \sum_{\substack{D~\mathrm{s.t.}\\ \sum_i d_i=d}} \sum_{\Tau} \mathrm{C}[x, \zeta^{D,\Tau}]
.\end{align*}
$$

[ja]: #
定義から $\mathrm{MC}=\mathrm{C}^1$ であるとわかります。
これが情報処理容量が記憶容量を拡張した指標であるとされる理由です。
さらに総容量 $C^\mathrm{tot}$ は以下の式で定義されます。

[en]: #
From the definition, we have $\mathrm{MC}=\mathrm{C}^1$.
This is why IPC is an extended metric of memory capacity.
The total capacity $\mathrm{C}^\mathrm{tot}$ is defined as:

[zh]: #
根据定义，我们有 $\mathrm{MC}=\mathrm{C}^1$。
这就是为什么 IPC 是记忆容量的扩展度量。
总容量$\mathrm{C}^\mathrm{tot}$定义为：

[END]: #
$$
\begin{align*}
\mathrm{C}^\mathrm{tot}[x, \zeta] &:= \sum_{d=1}^{\infty} \mathrm{C}^d[x, \zeta]
.\end{align*}
$$

[ja]: #
J. Dambreら<sup>[1]</sup>はこの情報処理容量の上限に関して以下の不等式を示しました (導出は発展課題)。

[en]: #
J. Dambre et al.<sup>[1]</sup> demonstrated the following inequality regarding the upper limit of this IPC (derivation is an advanced exercise):

[zh]: #
J. Dambre 等人<sup>[1]</sup> 证明了关于此 IPC 上限的以下不等式(推导是一项高级练习)：

[END]: #
$$
\begin{align*}
\mathrm{C}^\mathrm{tot}[x, \zeta] \leq r \leq N
,\end{align*}
$$

[ja]: #
ここで $r$ は力学系の階数を表します。
これは記憶容量同様に情報処理容量の上限が、内部状態の線形独立な次元数に制限される点を示しています。

この指標によって導かれた結果は特に物理リザバー計算 (Physical Reservoir Computing; PRC) において重要な示唆を与えます。
PRCでは通常設置されるセンサの数がそのまま内部状態の次元数に対応し、線形独立なセンサ時系列の数がそのまま情報処理容量の上限を表すからです。
また情報処理容量はどのような変換が行われているのか全網羅的に評価するため、物理系そのものの特性の評価にも役立ちます。

[en]: #
where $r$ represents the rank of the dynamical system.
This indicates that, similar to memory capacity, the upper limit of IPC is constrained by the number of linearly independent dimensions of the internal states.

The results derived from this metric provide particularly important insights for physical reservoir computing (PRC).
In PRC, the number of sensors typically installed directly corresponds to the dimensionality of the internal states, and the number of linearly independent sensor time series directly represents the upper limit of IPC.
Moreover, since IPC comprehensively evaluates what kinds of transformations are performed, it is also useful for assessing the characteristics of the physical system itself.

[zh]: #
其中 $r$ 表示动力系统的等级。
这表明，与记忆容量类似，IPC的上限受到内部状态线性独立维数的限制。

从该指标得出的结果为物理储池计算提供了特别重要的见解。
在PRC中，通常安装的传感器数量直接对应于内部状态的维数，线性独立传感器时间序列的数量直接代表了IPC的上限。
此外，由于IPC全面评估执行了哪些类型的转换，因此它对于评估物理系统本身的特性也很有用。

[ja]: #
## 演習問題と実演

[en]: #
## Exercises and demonstrations

[zh]: #
## 练习和演示

[ja]: #
ここからは演習問題とデモンストレーションに移ります。
前回と同じライブラリの他、前回の演習で実装した`ESN`・`Linear`・`narma_func`が`import`により利用できます。
初めに次のセルを実行してください。

なお`ESN`・`Linear`・`narma_func`の内部実装を再確認するには、`import inspect`以下の行をコメントアウトするか`...?? / ??...`を使用してください。

[en]: #
Let's move on to the exercises and demonstrations.
Along with the basic libraries from the previous chapter, you can import and use the `ESN`, `Linear`, and `narma_func` we implemented earlier.
Please run the following cell.

You can view the implementations of `ESN`, `Linear`, and `narma_func` by uncommenting the lines after `import inspect` or by using `...?? / ??...`.

[zh]: #
让我们继续进行练习和演示。
除了上一章中的基本库之外，您还可以导入和使用我们之前实现的 `ESN`、`Linear` 和 `narma_func`。
请运行以下单元格。

您可以通过取消注释 `import inspect` 后面的行或使用 `...?? / ??...` 来查看 `ESN`、`Linear` 和 `narma_func` 的实现。

In [ ]:
import itertools
import math
import sys

import numpy as np
import scipy as sp

if "google.colab" in sys.modules:
    from google.colab import drive  # type: ignore

    # NOTE: Set `save_to_drive` to True to save your progress to Google Drive.
    # After running this cell, you may need to reopen this notebook directly from Google Drive (https://drive.google.com).
    # Then, set both `save_to_drive` and `project_path` to your preferred values and run this cell again.
    save_to_drive = False  # @param {type:"boolean"}

    # NOTE: Specify the destination path for project extraction (Default: Google Drive root directory).
    project_path = ""  # @param {type:"string"}

    if save_to_drive:
        drive.mount("/content/gdrive")
        base_path = f"/content/gdrive/My Drive/{project_path}"
        print(f"Using Google Drive storage (Saving to {base_path}).")
        !mkdir -p "$base_path"
        %cd "$base_path"
    else:
        print("Using temporary Colab storage (Progress will not be saved to Drive).")
        %cd /content/

    print("Cloning the project repository...")
    !git clone --branch [[BRANCH_NAME]] https://github.com/rc-bootcamp/[[PROJECT_NAME]].git [[PROJECT_NAME]]-[[BRANCH_NAME]]
    %cd [[PROJECT_NAME]]-[[BRANCH_NAME]]
else:
    sys.path.append(".")

from ipc_module.helper import visualize_dataframe
from ipc_module.profiler import UnivariateProfiler, UnivariateViewer
from utils.reservoir import ESN, Linear
from utils.style_config import Figure, plt
from utils.tester import load_from_chapter_name
from utils.tqdm import tqdm, trange

test_func, show_solution = load_from_chapter_name("07_information_processing_capacity")

# Uncomment it to see the implementations of `Linear` and `ESN`.
# import inspect
# print(inspect.getsource(Linear))
# print(inspect.getsource(ESN))

# Or just use ??.../...?? (uncomment the following lines).
# Linear??
# ESN??

[ja]: #
### 1. ルジャンドル多項式と直交性の確認

[en]: #
### 1. Legendre polynomials and verification of orthogonality

[zh]: #
### 1. 勒让德多项式(Legendre polynomial)及正交性验证

[ja]: #
ここまで直交多項式を $\mathcal{P}$ とおいて説明しましたが、具体的な多項式を導入して議論しましょう。
実はこの直交多項式は入力時系列が従う分布に依存します。
例えば入力時系列 $\zeta[k]$ が 一様乱数 $\mathcal{U}([-1, 1])$ に従う場合、以下の漸化式で定義される[ルジャンドル多項式](https://ja.wikipedia.org/wiki/%E3%83%AB%E3%82%B8%E3%83%A3%E3%83%B3%E3%83%89%E3%83%AB%E5%A4%9A%E9%A0%85%E5%BC%8F)を使用できます。

[en]: #
So far, we have explained orthogonal polynomials denoted as $\mathcal{P}$, but let's introduce specific polynomials for discussion.
These orthogonal polynomials depend on the distribution of the input time series.
For example, if the input time series $\zeta[k]$ follows a uniform random distribution $\mathcal{U}([-1, 1])$, we can use the [Legendre polynomials](https://en.wikipedia.org/wiki/Legendre_polynomials) defined by the following recurrence relation:

[zh]: #
到目前为止，我们已经解释了表示为$\mathcal{P}$的正交多项式，但是我们引入具体的多项式来进行讨论。
这些正交多项式取决于输入时间序列的分布。
例如，如果输入时间序列 $\zeta[k]$ 服从均匀随机分布 $\mathcal{U}([-1, 1])$，我们可以使用由以下递推关系定义的 [勒让德多项式](https://en.wikipedia.org/wiki/Legendre_polynomials)：

[END]: #
$$
\begin{align*}
(n+1)\mathcal{P}_{n+1}(z) &= (2n+1)z\mathcal{P}_n(z) - n\mathcal{P}_{n-1}(z)
,\end{align*}
$$

[ja]: #
ただし$\mathcal{P}_0(z)=1,~\mathcal{P}_1(z)=z$ とします。
これを式展開すると以下のような多項式が得られます。

[en]: #
where $\mathcal{P}_0(z)=1$ and $\mathcal{P}_1(z)=z$.
Expanding this equation yields the following polynomials:

[zh]: #
其中 $\mathcal{P}_0(z)=1$ 和 $\mathcal{P}_1(z)=z$。
展开该方程可得到以下多项式：

[END]: #
$$
\begin{align*}
\mathcal{P}_0(z) &= 1 \\
\mathcal{P}_1(z) &= z \\
\mathcal{P}_2(z) &= \frac{1}{2}(3z^2-1) \\
\mathcal{P}_3(z) &= \frac{1}{2}(5z^3-3z) \\
\mathcal{P}_4(z) &= \frac{1}{8}(35z^4-30z^2+3) \\
\mathcal{P}_5(z) &= \frac{1}{8}(63z^5-70z^3+15z) \\
\mathcal{P}_6(z) &= \frac{1}{48}(231z^6-315z^4+105z^2-5)
.\end{align*}
$$

[ja]: #
一方で直交性は内積計算の結果により評価できます。
いま2つの目標時系列 $\zeta^A:=\zeta^{D_A,\Tau_A}, \zeta^B:=\zeta^{D_B,\Tau_B}$ からそれぞれ$T$ ステップ分抽出された2つの目標時系列行列 $\Zeta^A = [\zeta^A[0]; \zeta^A[1];~\ldots;~\zeta^A[T-1]] \in \mathbb{R}^{T \times 1}$ と $\Zeta^B = [\zeta^B[0]; \zeta^B[1];~\ldots;~\zeta^B[T-1]]  \in \mathbb{R}^{T \times 1}$を用意します。
 $\Zeta^A$と$\Zeta^B$の内積 $I(\Zeta^A, \Zeta^B)$ は以下のように計算できます。

[en]: #
Orthogonality, on the other hand, can be evaluated based on the result of inner product calculations.
Now, consider two target time series $\zeta^A := \zeta^{D_A,\Tau_A}$ and $\zeta^B := \zeta^{D_B,\Tau_B}$, from which two target time series matrices $\Zeta^A = [\zeta^A[0]; \zeta^A[1];~\ldots;~\zeta^A[T-1]] \in \mathbb{R}^{T \times 1}$ and $\Zeta^B = [\zeta^B[0]; \zeta^B[1];~\ldots;~\zeta^B[T-1]] \in \mathbb{R}^{T \times 1}$ are extracted for $T$ steps.
The inner product $I(\Zeta^A, \Zeta^B)$ of $\Zeta^A$ and $\Zeta^B$ can be calculated as follows:

[zh]: #
另一方面，正交性可以根据内积计算的结果来评估。
现在，考虑两个目标时间序列、$\zeta^A := \zeta^{D_A,\Tau_A}$ 和 $\zeta^B := \zeta^{D_B,\Tau_B}$，从中提取两个目标时间序列矩阵 $\Zeta^A = [\zeta^A[0]; \zeta^A[1];~\ldots;~\zeta^A[T-1]] \in \mathbb{R}^{T \times 1}$ 和 $\Zeta^B = [\zeta^B[0]; \zeta^B[1];~\ldots;~\zeta^B[T-1]] \in \mathbb{R}^{T \times 1}$ 用于 $T$ 步骤。
$\Zeta^A$和$\Zeta^B$的内积$I(\Zeta^A, \Zeta^B)$可以计算如下：

[END]: #
$$
\begin{align*}
I(\Zeta^A, \Zeta^B) &= \sum_{t=1}^{T} \frac{\Zeta^A_t}{\sqrt{\sum_{t=1}^T (\Zeta^A_t)^2}} \cdot \frac{\Zeta^B_t}{\sqrt{\sum_{t=1}^T (\Zeta^B_t)^2}}
.\end{align*}
$$

[ja]: #
直交性により $I$ の期待値は以下のとおり計算されます。

[en]: #
The expected value of $I$ based on orthogonality is calculated as follows:

[zh]: #
基于正交性的$I$的期望值计算如下：

[END]: #
$$
\begin{align*}
\mathrm{E}[I(\Zeta^A, \Zeta^B)] &= \begin{cases}
1 & \mathrm{if}~\zeta^A = \zeta^B~(\Leftrightarrow  D_A=D_B \land \Tau_A=\Tau_B) \\
0 & \mathrm{if}~\zeta^A \perp \zeta^B~(\mathrm{otherwise}) \\
\end{cases}
.\end{align*}
$$

[ja]: #
演習問題で実際にルジャンドル多項式と内積計算を実装し、これらを確認しましょう。

[en]: #
Let us implement the Legendre polynomials and inner product calculations in the exercises to verify these properties.

[zh]: #
让我们在练习中实现勒让德多项式和内积计算来验证这些属性。

Q1.1

[ja]: #
上記の漸化式を参考に、ルジャンドル多項式を計算するクラス`Legendre`内のメソッド`Legendre._calc`を完成させよ。
なお`Legendre._calc`は時系列 $\Zeta$ に対して $\mathcal{P}_n(\Zeta)$ を計算する。
結果は一様乱数 $\mathcal{U}([-1, 1])$ からサンプルされた長さ $T$ の時系列より結果は検証される。

[en]: #
Based on the above recurrence relation, complete the method `Legendre._calc` in the class `Legendre` to compute the Legendre polynomial. Note that `Legendre._calc` calculates $\mathcal{P}_n(\Zeta)$ for the time series $\Zeta$. The result will be validated against a time series of length $T$ sampled from a uniform random distribution $\mathcal{U}([-1, 1])$.

[zh]: #
根据上述递推关系，完成类`Legendre`中的方法`Legendre._calc`，计算出勒让德多项式。
请注意，`Legendre._calc` 计算时间序列 $\Zeta$ 的 $\mathcal{P}_n(\Zeta)$。
结果将根据从均匀随机分布 $\mathcal{U}([-1, 1])$ 中采样的长度为 $T$ 的时间序列进行验证。

[END]: #
- `Legendre._calc`
  - Argument(s):
    - `n`: `int`
      - `n >= 0`
  - Operation(s):
    - Update `self._cache[n]`
- $10 \leq T \leq 10^{3}$, $1\leq n \leq 20$

[tips]: #
$$
\begin{align*}
n\mathcal{P}_{n}(z) &= (2n-1)z\mathcal{P}_{n-1}(z) - (n-1)\mathcal{P}_{n-2}(z) ~\mathrm{for}~n \geq 2 \\
.\end{align*}
$$
[/tips]: #

In [ ]:
class Legendre(object):
    def __init__(self, xs):
        self.xs = xs
        self._caches = {}
        self._caches[0] = 1
        self._caches[1] = self.xs

    def __getitem__(self, deg):
        assert deg >= 0
        if deg not in self._caches:
            self._caches[deg] = self._calc(deg)
        return self._caches[deg]

    def _calc(self, n: int):
        # BEGIN Use `self.xs` and `self[n-1]`, `self[n-2]` to calculate the n-th Legendre polynomial.
        res = ((2 * n - 1) / n) * self.xs * self[n - 1]
        res -= ((n - 1) / n) * self[n - 2]
        return res
        # END


def solution(us, n):
    # DO NOT CHANGE HERE.
    poly = Legendre(us)
    return poly[n]


test_func(solution, "01_01")
# show_solution("01_01", "Legendre")  # Uncomment it to see the solution.

Q1.2.

[ja]: #
上の式に基づき、２つの時系列 $A \in \mathbb{R}^{T}$ と $B\in \mathbb{R}^{T}$ の内積を計算するメソッド`calc_inner_product`を完成させよ。

[en]: #
Based on the above equation, complete the method `calc_inner_product` to calculate the inner product of two time series $A \in \mathbb{R}^{T}$ and $B \in \mathbb{R}^{T}$.

[zh]: #
根据上式，完成方法`calc_inner_product`，计算两个时间序列、$A \in \mathbb{R}^{T}$和$B \in \mathbb{R}^{T}$的内积。

[END]: #

- `calc_inner_product`
  - Argument(s):
    - `a`: `np.ndarray`
      - `shape`: `(t,)`
      - `dtype`: `np.float64`
  - Return(s):
    - `b`: `np.ndarray`
      - `shape`: `(t,)`
      - `dtype`: `np.float64`
- $10 \leq T \leq 10^3$

In [ ]:
def calc_inner_product(a, b):
    # BEGIN Calculate and return the inner product between vectors a and b.
    return a.dot(b) / np.linalg.norm(a) / np.linalg.norm(b)
    # END


test_func(calc_inner_product, "01_02")
# show_solution("01_02")  # Uncomment it to see the solution.

[ja]: #
さて実際に多項式の直交性を確認しましょう。
まずは[Wikipediaのルジャンドル多項式の図表](https://en.wikipedia.org/wiki/Legendre_polynomials#/media/File:Legendrepolynomials6.svg)を正しく再現できるか確認します。

[en]: #
Now, let's verify their orthogonality.
First, let's check that we can reproduce the [figure of Legendre polynomials from Wikipedia](https://en.wikipedia.org/wiki/Legendre_polynomials#/media/File:Legendrepolynomials6.svg).

[zh]: #
现在，让我们验证它们的正交性。
首先，让我们检查一下是否可以重现[来自维基百科的勒让德多项式图](https://en.wikipedia.org/wiki/Legendre_polynomials#/media/File:Legendrepolynomials6.svg)。

In [ ]:
# https://en.wikipedia.org/wiki/Legendre_polynomials
us = np.linspace(-1, 1, 1000)
poly = Legendre(us)
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
for deg in range(1, 6):
    ax.plot(us, poly[deg], label=r"$\mathcal{P}_" + f"{{{deg}}}$", color=f"C{deg}")
ax.legend(
    loc="upper left",
    fontsize=12,
    bbox_to_anchor=(1.025, 1.0),
    borderaxespad=0,
    frameon=False,
)
ax.tick_params(axis="both", labelsize=12)

None

[ja]: #
次に一様乱数 $\mathcal{U}([-1, 1])$ によって時系列を生成し、時間遅れとルジャンドル多項式によって様々な目標時系列を生成し、それらの間の内積を計算してみましょう。
`degree_delay_list` の各要素は $\zeta^{D,\Tau}$ における次数の集合 $D$ と時間遅れの集合 $\Tau$ の組を指定します。
色々変えてみて直交性が保たれているか確認してみましょう。

[en]: #
Next, generate a time series using the uniform random distribution $\mathcal{U}([-1, 1])$, create various target time series using time delay and Legendre polynomials, and calculate the inner products between them.
Each element of `degree_delay_list` specifies a pair of degree sequence $D$ and time delay sequence $\Tau$ in $\zeta^{D,\Tau}$.
Try changing them in various ways and check if orthogonality is maintained.

[zh]: #
接下来，使用均匀随机分布$\mathcal{U}([-1, 1])$生成时间序列，使用时间延迟和勒让德多项式创建各种目标时间序列，并计算它们之间的内积。
`degree_delay_list`的每个元素指定$\zeta^{D,\Tau}$中的一对度序列$D$和时延序列$\Tau$。
尝试以各种方式改变它们并检查是否保持正交性。

In [ ]:
seed = 1234
t_washout, t_sample = 100, 10000

rnd = np.random.default_rng(seed)
t_total = t_washout + t_sample
us = rnd.uniform(-1, 1, t_total)
poly = Legendre(us)

degree_delay_list = [
    ([1], [0]),
    ([1], [10]),
    ([1, 1], [1, 2]),
    ([2], [0]),
    ([3], [5]),
]  # You can add or change more combinations if you want.


def create_poly(args):
    degrees, taus = args
    out = 1
    for deg, tau in zip(degrees, taus, strict=True):
        out *= poly[deg][t_washout - tau : t_total - tau]
    return out


def create_label(args):
    degrees, taus = args
    out = r"$"
    for d, t in zip(degrees, taus, strict=True):
        if t == 0:
            out += f"P_{{{d}}}(\\zeta)"
        else:
            out += f"P_{{{d}}}(\\zeta^{{{t}}})"
    out += r"$"
    return out


length = len(degree_delay_list)
polys = list(map(create_poly, degree_delay_list))
labels = list(map(create_label, degree_delay_list))
products = np.zeros((length, length))

for idx, idy in itertools.product(range(length), range(length)):
    products[idx, idy] = calc_inner_product(polys[idx], polys[idy])

fig = Figure(figsize=(8, 6))
ax = fig[0]
im, cb = ax.plot_matrix(
    products,
    cmap="Blues",
    vmin=0,
    vmax=1,
    aspect="equal",
    colorbar=True,
)
ax.set_xticks(range(length))
ax.set_yticks(range(length))
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticklabels(labels, fontsize=10)
cb.ax.tick_params(labelsize=12)

None

Q1.3 (Advanced)

[ja]: #
- 上のデモでは時系列の長さ`t_sample`によって内積の値が変化する。
時系列の長さが短いとき、直交していても$I$の値が0近くにならない点を確認せよ。
- 一様乱数$\mathcal{U}([-1, 1])$の代わりに標準正規分布 $\mathcal{N}(0, 1)$を用いる時代わりに[Hermite多項式](https://en.wikipedia.org/wiki/Hermite_polynomials#Recurrence_relation)を使用しなければならない<sup>[2]</sup>。
 クラス`Hermite`を実装し、Hermite多項式を計算できるようにし、同様に直交性を確認せよ。

[en]: #
- In the above demo, the value of the inner product $I$ changes depending on the time series length `t_sample`. Confirm that when the time series length is short, the value of $I$ does not approach 0 even if they are orthogonal.
- Instead of using the uniform random distribution $\mathcal{U}([-1, 1])$, when using the standard normal distribution $\mathcal{N}(0, 1)$, [Hermite polynomials](https://en.wikipedia.org/wiki/Hermite_polynomials#Recurrence_relation) must be used instead<sup>[2]</sup>. Implement the class `Hermite` to calculate Hermite polynomials and similarly verify their orthogonality.

[zh]: #
- 在上面的演示中，内积 $I$ 的值根据时间序列长度 `t_sample` 的变化而变化。
确认当时间序列长度较短时，即使它们正交，$I$的值也不会接近0。
- 当使用标准正态分布 $\mathcal{N}(0, 1)$ 时，必须使用 [Hermite 多项式](https://en.wikipedia.org/wiki/Hermite_polynomials#Recurrence_relation) 代替 <sup>[2]</sup>，而不是使用均匀随机分布 $\mathcal{U}([-1, 1])$。
实现类 `Hermite` 来计算 Hermite 多项式并类似地验证它们的正交性。

[ja]: #
### 2. 情報処理容量の実装と確認

[en]: #
### 2. Implementation and verification

[zh]: #
### 2. 实施与验证

[ja]: #
さて実際に情報処理容量を計算してみましょう。
まず $N=10$で活性化関数 $\tanh$ のESNと、一様乱数 $\mathcal{U}([-1, 1])$ に従う入力時系列 $\zeta[k]$ を用意し、その時のダイナミクス $x[k]$ をサンプルします。
非対称性を確保するため、$\zeta[k]$ は $[0, 1]$の範囲を取るようにスケーリングされます
(入力を非対称にしたのは$\tanh$が奇関数であるため、対称入力に対しては奇数次の成分の容量しか出現しないからです[確認は発展課題])。

[en]: #
Now, let's actually calculate the IPC.
First, prepare an ESN with $N=10$ and the activation function $\tanh$, along with an input time series $\zeta[k]$ that follows the uniform random distribution $\mathcal{U}([-1, 1])$, and sample its dynamics $x[k]$.
To ensure asymmetry, $\zeta[k]$ is scaled to take values in the range $[0, 1]$.
(The input is made asymmetric because $\tanh$ is an odd function, and for symmetric input, only the odd-order components of the capacity appear [verification is an advanced task]).

[zh]: #
现在，我们来实际计算一下IPC。
首先，准备一个带有 $N=10$ 和激活函数 $\tanh$ 的 ESN，以及遵循均匀随机分布 $\mathcal{U}([-1, 1])$ 的输入时间序列 $\zeta[k]$，并对其动态 $x[k]$ 进行采样。
为了确保不对称性，$\zeta[k]$ 被缩放以取 $[0, 1]$ 范围内的值。
(输入不对称，因为 $\tanh$ 是奇函数，对于对称输入，仅出现容量的奇数阶分量[验证是一项高级任务])。

In [ ]:
seed = 5678
dim = 10
t_washout = 1000
t_sample = 1000000
t_total = t_washout + t_sample
display = True

rnd = np.random.default_rng(seed)
w_in = Linear(1, dim, bound=0.1, bias=0.0, rnd=rnd)
net = ESN(dim, sr=0.1, f=np.tanh, p=1, rnd=rnd)

x0 = np.zeros((dim,))
us = rnd.uniform(-1, 1, (t_total, 1))

x = x0
xs = np.zeros((t_total, *x0.shape))
for idx in trange(t_total, display=display):
    x = net(x, w_in(0.5 * us[idx] + 0.5))
    # x = net(x, w_in(us[idx]))  # Uncomment it for the symmetric case.
    xs[idx] = x

print("us:", us.shape)
print("xs:", xs.shape)

[ja]: #
前章で扱われた記憶関数の計算同様に、情報処理容量もSVDを用いて計算します。
したがって同じコードを流用できます (確認していない人は前章に戻ってください) 。
以下の`calc_regression_and_rank`と`calc_capacity`は前章の実装を流用したものです。

[en]: #
As with the memory function calculations covered in the previous chapter, IPC is also computed using SVD.
Thus, the same code can be reused (if you haven't reviewed it, please refer back to the previous chapter).
The following `calc_regression_and_rank` and `calc_capacity` functions are reused implementations from the previous chapter.

[zh]: #
与上一章中介绍的记忆函数计算一样，IPC 也是使用 SVD 计算的。
因此，可以重用相同的代码(如果您还没有查看过，请参阅上一章)。
以下 `calc_regression_and_rank` 和 `calc_capacity` 函数是重用上一章的实现。

In [ ]:
def calc_regression_and_rank(X):
    T, N = X.shape[-2:]
    X = X - X.mean(axis=-2, keepdims=True)
    U, sigma, _V = np.linalg.svd(X, full_matrices=False)
    eps = np.finfo(X.dtype).eps
    sigma_sq_max = np.max(sigma * sigma, axis=-1, keepdims=True)
    eps = sigma_sq_max * (eps * max(T, N))
    mask = sigma > eps
    rank = mask.sum(axis=-1)
    return U, mask, rank


def calc_capacity(U, mask, zeta):
    uzeta = U.swapaxes(-2, -1) @ zeta
    dot = ((uzeta * uzeta) * mask[..., None]).sum(axis=-2)
    var = (zeta * zeta).sum(axis=-2)
    r2 = dot / var
    return r2

[ja]: #
まずSVDを実行し階数 $r$ を確認しましょう。
階数は $\mathrm{C}^\mathrm{tot}$ の上限となります。
また直交多項式 $\mathcal{P}$ も`Legendre`によって計算できるように準備しましょう。

[en]: #
First, let's perform SVD and check the rank $r$.
The rank serves as the upper limit of $\mathrm{C}^\mathrm{tot}$.

[zh]: #
首先，我们执行 SVD 并检查排名 $r$。
该等级作为$\mathrm{C}^\mathrm{tot}$的上限。

In [ ]:
U, mask, rank = calc_regression_and_rank(xs[t_washout:])
print("rank:", rank)

poly = Legendre(us)

[ja]: #
#### 1次の容量の計算

[en]: #
#### Calculation of first-order capacity

[zh]: #
#### 一阶容量计算

[ja]: #
まずは1次の容量 $C^1$ を計算しましょう。
特に $\mathcal{P}_1(z) = z$ なので1次の目標時系列は以下のとおり計算できます。

[en]: #
First, let's calculate the first-order capacity $C^1$.
Specifically, since $\mathcal{P}_1(z) = z$, the first-order target time series can be calculated as follows:

[zh]: #
首先我们计算一阶容量$C^1$。
具体来说，从$\mathcal{P}_1(z) = z$开始，一阶目标时间序列可以计算如下：

[END]: #
$$
\begin{align*}
\zeta^{0}, \zeta^{1}, \zeta^{2}, \zeta^{3},~\ldots
.\end{align*}
$$

In [ ]:
delay1_max = 10

taus = np.arange(0, delay1_max + 1)
c1 = np.zeros(len(taus))
for idx, tau in enumerate(tqdm(taus)):
    zeta = poly[1][t_washout - tau : t_total - tau]
    c1[idx] = calc_capacity(U, mask, zeta)[..., 0]

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(taus, c1, label=r"$\zeta^1$", color="C0", marker="o")
ax.set_xlim(-0.5, delay1_max + 0.5)
ax.set_xlabel(r"$\tau$", fontsize=14)
ax.set_ylabel(r"$\mathrm{C}[x,\zeta^\tau]$", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=12)
ax.set_title(r"$\mathrm{C}^1$=" + f"{c1.sum():.3f}", fontsize=14)

None

[ja]: #
#### 2次の容量の計算

[en]: #
#### Calculation of second-order capacity

[zh]: #
#### 二阶容量计算

[ja]: #
2次の場合は $D=\{1,1\}$ と $D=\{2\}$ の2パターンの組み合わせが考えられます。
$\tau_1 \leq \tau_2$となる $\tau_1$ と $\tau_2$ を用意し以下の形で2次の目標時系列 $z^{\tau_1, \tau_2} $ を網羅的に用意できます。

[en]: #
In the case of the second order, there are two possible combinations: $D=\{1,1\}$ and $D=\{2\}$.
Using $\tau_1$ and $\tau_2$ such that $\tau_1 \leq \tau_2$, we can comprehensively prepare the orthogonal polynomials $z^{\tau_1, \tau_2}$ in the following form:

[zh]: #
对于第二阶，有两种可能的组合：$D=\{1,1\}$ 和 $D=\{2\}$。
利用$\tau_1$和$\tau_2$，即$\tau_1 \leq \tau_2$，我们可以综合制备正交多项式 $z^{\tau_1, \tau_2}$，其形式如下：

[END]: #
$$
\begin{align*}
z^{\tau_1, \tau_2} = \begin{cases}
\mathcal{P}_2(\zeta^{\tau_1}) &= \frac{3}{2}(\zeta^{\tau_1})^2 - \frac{1}{2} & \mathrm{if}~\tau_1 = \tau_2 \\
\mathcal{P}_1(\zeta^{\tau_1})\mathcal{P}_1(\zeta^{\tau_2}) &= \zeta^{\tau_1} \zeta^{\tau_2} & \mathrm{if}~\tau_1 \leq \tau_2 \\
\end{cases}
.\end{align*}
$$

In [ ]:
delay2_max = 8

taus = np.arange(0, delay2_max + 1)
c2 = np.zeros((len(taus), len(taus)))

cands = list(itertools.product(enumerate(taus), repeat=2))
for (idx, tau1), (idy, tau2) in tqdm(cands):
    if not (idx <= idy):
        continue
    if idx == idy:
        zeta = poly[2][t_washout - tau1 : t_total - tau1]
    else:
        zeta1 = poly[1][t_washout - tau1 : t_total - tau1]
        zeta2 = poly[1][t_washout - tau2 : t_total - tau2]
        zeta = zeta1 * zeta2
    c2[idx, idy] = calc_capacity(U, mask, zeta)[..., 0]

fig = Figure(figsize=(8, 6))
ax = fig[0]
im, cb = ax.plot_matrix(
    c2,
    x=taus,
    y=taus,
    cmap="viridis",
    zscale="log",
    vmax=1,
    vmin=1e-3,
    aspect="equal",
    colorbar=True,
    xticks_kws=dict(num_tick=len(taus)),
    yticks_kws=dict(num_tick=len(taus)),
)
ax.grid(False)
ax.set_xlim(-1, len(taus))
ax.set_ylim(-1, len(taus))
ax.set_xlabel(r"$\tau_2$", fontsize=14)
ax.set_ylabel(r"$\tau_1$", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=12)
cb.ax.tick_params(labelsize=12)
cb.set_label(r"$\mathrm{C}[x,z^{\tau_1,\tau_2}]$", fontsize=14)
ax.set_title(r"$\mathrm{C}^2$=" + f"{c2.sum():.3f}", fontsize=14)

None

[ja]: #
#### 3次の容量の計算

[en]: #
#### Calculation of third-order capacity

[zh]: #
#### 三阶容量计算

[ja]: #
3次の場合は $D=\{1,1,1\}$ と $D=\{2, 1\}$ ならびに $D=\{3\}$ の3パターンの組み合わせが考えられます。
2次の場合同様に$\tau_1 \leq \tau_2 \leq \tau_3$となる $\tau_1,\tau_2,\tau_3$ を用意し以下の形で3次の目標時系列 $z^{\tau_1, \tau_2,\tau_3} $ を網羅的に用意できます。

[en]: #
For the third order, there are three possible combinations: $D=\{1,1,1\}$, $D=\{2, 1\}$, and $D=\{3\}$.
Similar to the second-order case, we prepare $\tau_1, \tau_2, \tau_3$ such that $\tau_1 \leq \tau_2 \leq \tau_3$ and construct the third-order target time series $z^{\tau_1, \tau_2, \tau_3}$ as follows:

[zh]: #
对于第三阶，有三种可能的组合：$D=\{1,1,1\}$、$D=\{2, 1\}$ 和 $D=\{3\}$。
与二阶情况类似，我们准备 $\tau_1, \tau_2, \tau_3$，使得 $\tau_1 \leq \tau_2 \leq \tau_3$ 并构造三阶目标时间序列 $z^{\tau_1, \tau_2, \tau_3}$，如下所示：

[END]: #
$$
\begin{align*}
z^{\tau_1, \tau_2, \tau_3} = \begin{cases}
\mathcal{P}_3(\zeta^{\tau_1}) &= \frac{5}{2}(\zeta^{\tau_1})^3 - \frac{3}{2}\zeta^{\tau_1} & \mathrm{if}~\tau_1 = \tau_2 = \tau_3 \\
\mathcal{P}_2(\zeta^{\tau_1})\mathcal{P}_1(\zeta^{\tau_3}) &= \left(\frac{3}{2}(\zeta^{\tau_1})^2 - \frac{1}{2}\right)\zeta^{\tau_3} & \mathrm{if}~\tau_1 = \tau_2 < \tau_3 \\
\mathcal{P}_1(\zeta^{\tau_1})\mathcal{P}_2(\zeta^{\tau_2}) &= \zeta^{\tau_1}\left(\frac{3}{2}(\zeta^{\tau_2})^2 - \frac{1}{2}\right) & \mathrm{if}~\tau_1 < \tau_2 = \tau_3 \\
\mathcal{P}_1(\zeta^{\tau_1})\mathcal{P}_1(\zeta^{\tau_2})\mathcal{P}_1(\zeta^{\tau_3}) &= \zeta^{\tau_1}\zeta^{\tau_2}\zeta^{\tau_3} & \mathrm{if}~\tau_1 < \tau_2 < \tau_3 \\
\end{cases}
.\end{align*}
$$

In [ ]:
delay3_max = 6

taus = np.arange(0, delay3_max + 1)
c3 = np.zeros((len(taus), len(taus), len(taus)))

cands = list(itertools.product(enumerate(taus), repeat=3))
for (idx, tau1), (idy, tau2), (idz, tau3) in tqdm(cands):
    if not (idx <= idy <= idz):
        continue
    if idx == idy == idz:
        zeta = poly[3][t_washout - tau1 : t_total - tau1]
    elif idx == idy:
        zeta1 = poly[2][t_washout - tau1 : t_total - tau1]
        zeta2 = poly[1][t_washout - tau3 : t_total - tau3]
        zeta = zeta1 * zeta2
    elif idy == idz:
        zeta1 = poly[1][t_washout - tau1 : t_total - tau1]
        zeta2 = poly[2][t_washout - tau2 : t_total - tau2]
        zeta = zeta1 * zeta2
    else:
        zeta1 = poly[1][t_washout - tau1 : t_total - tau1]
        zeta2 = poly[1][t_washout - tau2 : t_total - tau2]
        zeta3 = poly[1][t_washout - tau3 : t_total - tau3]
        zeta = zeta1 * zeta2 * zeta3
    c3[idx, idy, idz] = calc_capacity(U, mask, zeta)[..., 0]


num_col = math.ceil(len(taus) / 2)
grid_size = (2, num_col)
fig = Figure(figsize=(grid_size[1] * 3, grid_size[0] * 3))
fig.create_grid(*grid_size, hspace=0.35, wspace=0.3)

for pos in range(len(taus)):
    ax = fig[pos // num_col, pos % num_col]
    res = ax.plot_matrix(
        c3[pos],
        x=taus,
        y=taus,
        cmap="viridis",
        zscale="log",
        vmax=1,
        vmin=1e-3,
        aspect="equal",
        colorbar=len(taus) == (pos + 1),
        xticks_kws=dict(num_tick=len(taus)),
        yticks_kws=dict(num_tick=len(taus)),
    )
    ax.grid(False)
    ax.set_xlim(-1, len(taus))
    ax.set_ylim(-1, len(taus))
    if pos % num_col == 0:
        ax.set_ylabel(r"$\tau_2$", fontsize=14)
    if (pos // num_col) == grid_size[0] - 1:
        ax.set_xlabel(r"$\tau_3$", fontsize=14)
    ax.tick_params(axis="both", which="major", labelsize=12)
    ax.set_title(r"$\tau_1$=" + f"{taus[pos]}", fontsize=14)
    if len(taus) == (pos + 1):
        cb = res[1]
        cb.ax.set_position([0.9, 0.1, 0.03, 0.8])
        cb.ax.tick_params(labelsize=12)
        cb.set_label(r"$\mathrm{C}[x,z^{\tau_1,\tau_2,\tau_3}]$", fontsize=14)
if len(taus) < (grid_size[0] * grid_size[1]):
    for pos in range(len(taus), grid_size[0] * grid_size[1]):
        fig.delaxes(fig[pos // num_col, pos % num_col])
fig.suptitle(r"$\mathrm{C}^3$=" + f"{c3.sum():.3f}", fontsize=16)

None

[ja]: #
ここまで計算した $\mathrm{C}^1, \mathrm{C}^2, \mathrm{C}^3$ を足し合わせて、全体の情報処理容量 $\mathrm{C}^\mathrm{tot}$ を計算します。
これがほぼ階数 $r$ に等しくなる点を確認してください。

[en]: #
Finally, we sum up the calculated $\mathrm{C}^1, \mathrm{C}^2, \mathrm{C}^3$ to compute the overall IPC $\mathrm{C}^\mathrm{tot}$.

[zh]: #
最后，我们将计算出的$\mathrm{C}^1, \mathrm{C}^2, \mathrm{C}^3$相加，计算出整体IPC $\mathrm{C}^\mathrm{tot}$。

In [ ]:
c_tot = np.sum(c1) + np.sum(c2) + np.sum(c3)
print("total_capacity", c_tot, "rank", rank)

Q2.1. (Advanced)

[ja]: #
- 文献[1]を読み、$\mathrm{C}^\mathrm{tot} \leq r$ の導出を確認せよ。
- 入力を対称的にした場合、$\mathrm{C}^d$ の偶数次成分 (特に $d=2$) の消失を確認せよ。
またその理由も考察し説明せよ。
- 活性化関数を偶関数に変更し、その挙動も同様に確認せよ。

[en]: #
- Read reference [1] and verify the derivation of $\mathrm{C}^\mathrm{tot} \leq r$.
- Verify that the even-order components of $\mathrm{C}^d$ (particularly $d=2$) vanish when the input is made symmetric. Also, consider and explain the reason for this.
- Change the activation function to an even function and similarly verify its behavior.

[zh]: #
- 阅读参​​考文献 [1] 并验证 $\mathrm{C}^\mathrm{tot} \leq r$ 的推导。
- 验证当输入对称时，$\mathrm{C}^d$(特别是 $d=2$)的偶数阶分量消失。
另外，考虑并解释其原因。
- 将激活函数更改为偶函数并以类似方式验证其行为。

Q2.2. (Advanced)

[ja]: #
- ある整数$d$に対して$\sum_{i} d_i = d$ となるような次数の集合$D=\{d_i\}_i$を効率的に出力するコードを実装せよ (Cf. [ヤング図形](https://ja.wikipedia.org/wiki/%E3%83%A4%E3%83%B3%E3%82%B0%E5%9B%B3%E5%BD%A2))。
    - あるいは後述の`ipc-module`において実装されている[`make_degree_list`](https://github.com/rc-bootcamp/ipc-module/blob/main/src/ipc_module/helper.py#L60)を使用しても良い。
    - `from ipc_module.helper import make_degree_list`によってインポートして使用できる。
- ある次数の集合 $D$ と最大時間遅れ$\tau_\mathrm{max} \geq 1$ が与えられた時、$\tau_\mathrm{max}$以下の範囲で可能な時間遅れの組み合わせ $\Tau=\{\tau_i\}_i$ をすべて列挙するコードを実装せよ。

[en]: #
- Implement code to efficiently output a set of degrees $D=\{d_i\}_i$ such that $\sum_{i} d_i = d$ for a given integer $d$ (Cf. [Young tableau](https://en.wikipedia.org/wiki/Young_tableau)).
    - Alternatively, you may use the [`make_degree_list`](https://github.com/rc-bootcamp/ipc-module/blob/main/src/ipc_module/helper.py#L60) implemented in the `ipc-module` mentioned later.
    - It can be imported and used with `from ipc_module.helper import make_degree_list`.
- Implement code to enumerate all possible combinations of time delays $\Tau=\{\tau_i\}_i$ within the range of $\tau_\mathrm{max} \geq 1$, given a set of degrees $D$ and a maximum time delay $\tau_\mathrm{max}$.

[zh]: #
- 实现代码以有效地输出一组度数 $D=\{d_i\}_i$，使得对于给定整数 $d$ 为 $\sum_{i} d_i = d$(参见 [Young tableau](https://en.wikipedia.org/wiki/Young_tableau))。
    - 或者，您可以使用后面提到的 `ipc-module` 中实现的 [`make_degree_list`](https://github.com/rc-bootcamp/ipc-module/blob/main/src/ipc_module/helper.py#L60)。
    - 可与`from ipc_module.helper import make_degree_list`一起导入并使用。
- 给定一组度数 $D$ 和最大时间延迟 $\tau_\mathrm{max}$，实现代码以枚举 $\tau_\mathrm{max} \geq 1$ 范围内时间延迟 $\Tau=\{\tau_i\}_i$ 的所有可能组合。

[ja]: #
### 3. ライブラリを使用した高速な演算

[en]: #
### 3. Fast IPC computation using libraries

[zh]: #
### 3. 使用库进行快速 IPC 计算

[ja]: #
#### 環境の設定

[en]: #
#### Environmental setup

[zh]: #
#### 环境设置

[ja]: #
ここまで確認したとおり、情報処理容量は直交多項式と時間遅れの組み合わせを網羅的に探索する必要があるため、次数が大きくなると計算量が爆発的に増加します。
また次数が大きくなると次数の分割の仕方が指数的に増えるため、前節でのやり方のように逐一実装するのはとても大変です (分割の数の一般項は $p(n)\sim\frac{1}{4\sqrt{3}n}e^{\pi\sqrt{\frac{2n}{3}}}$ に漸近すると知られています<sup>[3]</sup>) 。
そのためこの演習では研究室内で開発されたライブラリ [`ipc-module`](https://rc-bootcamp.github.io/ipc-module/) を使い、情報処理能力を計算する方法を学びましょう。

`ipc-module`は`numpy`の他`pytorch`や`cupy`といったGPUを用いたテンソル演算のライブラリをサポートしているため、CPUのみでの演算と比べてより効率的に計算できます。
また整理や描画のための関数が用意されており、簡単に所望の力学系に対してその情報処理の内実を確認できます。
[PyPIにおいて公開](https://pypi.org/project/ipc-module/)されており、`pip install ipc-module`でインストールできますが、このノートブックでは手元で確認し変更を加えやすいようにソースコードを直接取り込んでいます。
具体的な実装は`./ipc_module`フォルダに入っているPythonコードを参照してください。

なお以下のコードはそのままCPU上でも動きますがかなり時間がかかります。
したがってGPUを使える環境にある場合は**GPUの使用を強く推奨します** (手元にない場合はGoogle Colaboratory上での実行をおすすめします)。
その際は以下のガイドを参考に追加の設定を行ってください。

[en]: #
As confirmed so far, calculating the IPC requires an exhaustive search over combinations of orthogonal polynomials and time delays, leading to an explosive increase in computational cost as the degree grows.
Additionally, as the degree increases, the number of ways to partition the degree grows exponentially, making it very challenging to implement each case individually as done in the previous section (it is known that the general term for the number of partitions asymptotically approaches $p(n)\sim\frac{1}{4\sqrt{3}n}e^{\pi\sqrt{\frac{2n}{3}}}$<sup>[3]</sup>).
Therefore, in this exercise, we will learn how to calculate IPC using the library [`ipc-module`](https://rc-bootcamp.github.io/ipc-module/), which was developed within the laboratory.

`ipc-module` supports GPU-accelerated tensor computation libraries such as `pytorch` and `cupy` in addition to `numpy`, enabling more efficient calculations compared to CPU-only operations.
It also provides functions for organizing and visualizing results, allowing you to easily examine the IPC of a given dynamical system.
It is [published on PyPI](https://pypi.org/project/ipc-module/) and can be installed via `pip install ipc-module`, but in this notebook, we directly include the source code to make it easier to check and modify locally.
For specific implementations, refer to the Python code in the `./ipc_module` folder.

Note that the following code will run on a CPU as is, but it will take considerable time.
Therefore, if you have access to a GPU environment, **using a GPU is strongly recommended** (if you don't have one locally, we recommend running it on Google Colaboratory).
In that case, follow the guide below for additional setup.

[zh]: #
目前已证实，计算IPC需要对正交多项式和时间延迟的组合进行详尽的搜索，导致计算成本随着度数的增长而爆炸性增加。
此外，随着度数的增加，划分度数的方法数量呈指数级增长，这使得像上一节中那样单独实现每种情况变得非常具有挑战性(众所周知，划分数量的通用术语渐近接近$p(n)\sim\frac{1}{4\sqrt{3}n}e^{\pi\sqrt{\frac{2n}{3}}}$<sup>[3]</sup>)。
因此，在本练习中，我们将学习如何使用实验室内开发的库 [`ipc-module`](https://rc-bootcamp.github.io/ipc-module/) 计算 IPC。

除了`numpy`之外，`ipc-module`还支持GPU加速的张量计算库，例如`pytorch`和`cupy`，与仅CPU运算相比，可以实现更高效的计算。
它还提供组织和可视化结果的功能，使您可以轻松检查给定动力系统的 IPC。
它是[发布在PyPI上](https://pypi.org/project/ipc-module/)，可以通过`pip install ipc-module`安装，但在这个笔记本中，我们直接包含源代码，以便更容易在本地检查和修改。
具体实现请参考`./ipc_module`文件夹中的Python代码。

请注意，以下代码将按原样在 CPU 上运行，但这将花费相当长的时间。
因此，如果您能够访问 GPU 环境，**强烈建议使用 GPU**(如果您本地没有 GPU，我们建议在 Google Colaboratory 上运行)。
在这种情况下，请按照以下指南进行其他设置。

[ja]: #
<details><summary>GPU環境で計算する場合の下準備</summary>

`pytorch` がとても便利ですので、以下その導入方法を説明します。

1. オンライン環境 (Google Colaboratory) の場合
    無料版でもデフォルトでGPUを使用できる他、`pytorch` がすでにインストールされているので特に追加の設定をする必要はないですが、以下の手順でGPUが有効か確認できます。
    「編集」 > 「ノートブックの設定」 > 「ハードウェア アクセラレータ」 > 「GPU」

2. ローカル環境の場合 (uvの場合)
    NVIDIA製のGPUの場合ドライバーをまずインストールしてください。
    インストールされているかどうかは、`nvidia-smi` コマンドで確認できます。
    インストールされていない場合は、公式の[配布ページ](https://www.nvidia.com/en-us/drivers/)からダウンロードできます。
    あとは以下のコマンドでインストールできます。
    ```bash
    uv sync --extra gpu
    ```
    自動的に`pytorch` のインストールが開始されます。

</details>

[en]: #
<details><summary>Preparations for calculations in a GPU environment</summary>

`pytorch` is very convenient, so the following explains how to set it up.

1. In an online environment (Google Colaboratory):
    Even with the free version, GPU can be used by default, and `pytorch` is already installed, so no additional setup is required. However, you can verify that the GPU is enabled using the following steps:
    "Edit" > "Notebook settings" > "Hardware accelerator" > "GPU"

2. In a local environment (for uv):
    For NVIDIA GPUs, first install the drivers.
    You can check if they are installed using the `nvidia-smi` command.
    If not installed, you can download them from the official [distribution page](https://www.nvidia.com/en-us/drivers/).
    Then, you can install them using the following command:
    ```bash
    uv sync --extra gpu
    ```
    This will automatically start the installation of `pytorch`.

</details>

[zh]: #
<details><summary>GPU环境计算的准备工作</summary>

`pytorch`非常方便，下面介绍如何设置。

1. 在线环境下(Google Colaboratory)：
    即使是免费版本，默认情况下也可以使用GPU，并且`pytorch`已经安装，所以不需要额外的设置。
    但是，您可以使用以下步骤验证 GPU 是否已启用：
    “编辑”>“笔记本设置”>“硬件加速器”>“GPU”

2. 在本地环境中(对于紫外线)：
    对于 NVIDIA GPU，首先安装驱动程序。
    您可以使用 `nvidia-smi` 命令检查它们是否已安装。
    如果未安装，您可以从官方[发行页面](https://www.nvidia.com/en-us/drivers/)下载它们。
    然后，您可以使用以下命令安装它们：
    ```bash
    uv sync --extra gpu
    ```
    这将自动开始安装 `pytorch`。

[ja]: #
#### 基本的な動作の説明

[en]: #
#### Explanation of basic operations

[zh]: #
#### 基本操作说明

[ja]: #
準備ができたら実際に`ipc_module`を使って計算してみましょう。
まずは$N=50$ 次元のESNを用意し、スペクトル半径を0.1から1.7の範囲で0.1刻みでふり、同時にそのダイナミクスをサンプルしましょう。
まずは先程同様に、一様乱数 $\mathcal{U}([-1, 1])$ から入力時系列 $\zeta$ を用意し、$[0, 1]$の範囲にスケーリングして非対称にしたものをESNに入力します (メモリ要求量が大きいので、メモリ不足のエラーが出たら適宜`t_sample`や`dim`を小さくしてください。
ただし一般にサンプルの長さが大きいほど計算の精度が上がります)。

[en]: #
Once ready, let's perform calculations using `ipc_module`.
First, prepare an ESN with $N=50$ dimensions, vary the spectral radius from 0.1 to 1.7 in increments of 0.1, and simultaneously sample its dynamics.
As before, prepare an input time series $\zeta$ from a uniform random distribution $\mathcal{U}([-1, 1])$, scale it to the range $[0, 1]$ to make it asymmetric, and input it into the ESN.
(The memory requirement is large, so if you encounter a memory shortage error, reduce `t_sample` or `dim` as needed.
In general, longer sample lengths lead to higher calculation accuracy.)

[zh]: #
准备好后，让我们使用 `ipc_module` 进行计算。
首先，准备一个 $N=50$ 尺寸的 ESN，将谱半径从 0.1 变化到 1.7，增量为 0.1，同时对其动态进行采样。
与之前一样，从均匀随机分布 $\mathcal{U}([-1, 1])$ 中准备输入时间序列 $\zeta$，将其缩放到范围 $[0, 1]$ 以使其不对称，并将其输入到 ESN 中。
(内存需求较大，因此如果遇到内存不足错误，请根据需要减少`t_sample`或`dim`。
一般来说，样本长度越长，计算精度越高。
)

In [ ]:
seed = 5678
dim = 50
t_washout = 10000
t_sample = 100000
srs = np.linspace(0.1, 1.7, 17)
t_total = t_washout + t_sample
display = True

rnd = np.random.default_rng(seed)
w_in = Linear(1, dim, bound=0.1, bias=0.0, rnd=rnd)
net = ESN(dim, sr=srs[:, None], f=np.tanh, p=1, rnd=rnd)

x0 = np.zeros((srs.shape[0], dim))
us = rnd.uniform(-1, 1, (t_total, 1))

x = x0
xs = np.zeros((t_total, *x0.shape))
for idx in trange(t_total, display=display):
    x = net(x, w_in(0.5 * us[idx] + 0.5))
    xs[idx] = x

print("us:", us.shape)
print("xs:", xs.shape)

[ja]: #
[`UnivariateProfiler`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateProfiler)は情報処理容量を計算する様々なメソッドを備えたクラスです。
1次元の入力時系列と対応する状態時系列を渡し、その後[`UnivariateProfiler.calc`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateProfiler.calc)に指定された次数と時間遅れの範囲で情報処理容量を計算します。

<details><summary> 引数の詳細</summary>

- `us`: `np.ndarray | torch.Tensor | cupy.ndarray`
    - 入力時系列
    - 形は `(t, ..., 1)` である必要がある
- `xs`: `np.ndarray | torch.Tensor | cupy.ndarray`
    - 対応する状態時系列
    - 形は `(t, ..., N)` である必要がある
- `poly_name`: `str`
    - 使用する多項式の名前
        - [`Legendre`](https://rc-bootcamp.github.io/ipc-module/polynomial/#ipc_module.polynomial.Legendre): ルジャンドル多項式
        - [`Hermite`](https://rc-bootcamp.github.io/ipc-module/polynomial/#ipc_module.polynomial.Hermite): エルミート多項式
        - [`GramSchmidt`](https://rc-bootcamp.github.io/ipc-module/polynomial/#ipc_module.polynomial.GramSchmidt): グラム・シュミット法による多項式展開
- `offset`: `int`
    - 時間遅れのオフセット ($t=0$ となるインデックスの指定)
    - デフォルトは0
- `surrogate_num`: `int`
    - サロゲートサンプルの数
    - デフォルトは1000
- `surrogate_seed`: `int`
    - サロゲートサンプルのシード
    - デフォルトは0
- `axis1`: `int`
    - 時間軸に対応するaxis
    - `us` と `xs` で同じものが使用される
    - デフォルトは0
- `axis2`: `int`
    - 状態に対応するaxis
    - `us` と `xs` で同じものが使用される
    - デフォルトは-1
</details>

実際に使ってみましょう。
まず `UnivariateProfiler`クラスのインスタンス`profiler`を作成します。
GPUが使えない環境の場合は`use_gpu`を`False`にしてください。

[en]: #
The [`UnivariateProfiler`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateProfiler) is a class equipped with various methods for calculating IPC.
You pass a one-dimensional input time series and the corresponding state time series, and then calculate the IPC within the specified range of degrees and time delays using [`UnivariateProfiler.calc`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateProfiler.calc).

<details><summary> Details of the arguments</summary>

- `us`: `np.ndarray | torch.Tensor | cupy.ndarray`
    - Input time series
    - Must have the shape `(t, ..., 1)`
- `xs`: `np.ndarray | torch.Tensor | cupy.ndarray`
    - Corresponding state time series
    - Must have the shape `(t, ..., N)`
- `poly_name`: `str`
    - Name of the polynomial to use
        - [`Legendre`](https://rc-bootcamp.github.io/ipc-module/polynomial/#ipc_module.polynomial.Legendre): Legendre polynomial
        - [`Hermite`](https://rc-bootcamp.github.io/ipc-module/polynomial/#ipc_module.polynomial.Hermite): Hermite polynomial
        - [`GramSchmidt`](https://rc-bootcamp.github.io/ipc-module/polynomial/#ipc_module.polynomial.GramSchmidt): Polynomial expansion using the Gram-Schmidt method
- `offset`: `int`
    - Time delay offset (specifies the index where $t=0$)
    - Default is 0
- `surrogate_num`: `int`
    - Number of surrogate samples
    - Default is 1000
- `surrogate_seed`: `int`
    - Seed for surrogate samples
    - Default is 0
- `axis1`: `int`
    - Axis corresponding to the time dimension
    - The same axis is used for both `us` and `xs`
    - Default is 0
- `axis2`: `int`
    - Axis corresponding to the state dimension
    - The same axis is used for both `us` and `xs`
    - Default is -1
</details>

Let’s try using it.
First, create an instance of the `UnivariateProfiler` class, named `profiler`.
If you are in an environment where a GPU cannot be used, set `use_gpu` to `False`.

[zh]: #
[`UnivariateProfiler`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateProfiler) 是一个配备了各种计算 IPC 的方法的类。
您传递一维输入时间序列和相应的状态时间序列，然后使用 [`UnivariateProfiler.calc`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateProfiler.calc) 计算指定度数和时间延迟范围内的 IPC。

<details><summary> 参数详细信息</summary>

- `us`：`np.ndarray | torch.Tensor | cupy.ndarray`
    - 输入时间序列
    - 形状必须为 `(t,..., 1)`
- `xs`：`np.ndarray | torch.Tensor | cupy.ndarray`
    - 对应状态时间序列
    - 形状必须为 `(t,..., N)`
- `poly_name`：`str`
    - 要使用的多项式的名称
        - [`Legendre`](https://rc-bootcamp.github.io/ipc-module/polynomial/#ipc_module.polynomial.Legendre): 勒让德多项式
        - [`Hermite`](https://rc-bootcamp.github.io/ipc-module/polynomial/#ipc_module.polynomial.Hermite)：厄米多项式
        - [`GramSchmidt`](https://rc-bootcamp.github.io/ipc-module/polynomial/#ipc_module.polynomial.GramSchmidt)：使用 Gram-Schmidt 方法进行多项式展开
- `offset`: `int`
    - 时间延迟偏移(指定$t=0$所在的索引)
    - 默认为 0
- `surrogate_num`：`int`
    - 替代样本的数量
    - 默认为 1000
- `surrogate_seed`：`int`
    - 替代样本的种子
    - 默认为 0
- `axis1`: `int`
    - 时间维度对应的轴
    - `us` 和 `xs` 使用相同的轴
    - 默认为 0
- `axis2`: `int`
    - 状态维度对应的轴
    - `us` 和 `xs` 使用相同的轴
    - 默认为-1

让我们尝试使用它。
首先，创建 `UnivariateProfiler` 类的实例，命名为 `profiler`。
如果您处于无法使用GPU的环境中，请将`use_gpu`设置为`False`。

In [ ]:
use_gpu = True  # NOTE: Set it to False to run on CPU.

if use_gpu:
    import torch

    assert torch.cuda.is_available(), "CUDA is not available"
    us_c = torch.from_numpy(us).cuda()
    xs_c = torch.from_numpy(xs).cuda()
    args = (us_c, xs_c)
else:
    args = (us, xs)

profiler = UnivariateProfiler(
    *args,
    "Legendre",
    offset=t_washout,
    surrogate_num=1000,
    axis1=0,
    axis2=-1,
)

[ja]: #
以下簡単なアルゴリズムの説明をします。
1. `UnivariateProfiler`クラスのインスタンス作成時に、第2引数に与えられた状態時系列 (今回の場合 `xs`) を正規化した後、SVDを実行し同時に階数を計測します (実装は`calc_regression_and_rank`とほぼ同じ)。
2. その直後に、サロゲートデータ (後述) に用いるシャッフルされたインデックスを生成します (`surrogate_num` で指定)。
3. `UnivariateProfiler.calc`メソッドを実行し、SVDの計算結果を基に指定された次数と時間遅れの範囲で情報処理容量を計算します。
目標時系列 (の構成に必要な直交多項式) の計算は遅延評価、すなわち必要になった際に計算されかつ結果がキャッシュされます。

1と2はすでに前のセルで完了したので以下のセルで `UnivariateProfiler.calc`メソッドを実行して、まず1次の容量 $C^1$ を計算してみましょう。

[en]: #
Here is a simple explanation of the algorithm:
1. When creating an instance of the `UnivariateProfiler` class, the state time series provided as the second argument (in this case, `xs`) is normalized, followed by performing SVD and simultaneously measuring the rank (the implementation is almost the same as `calc_regression_and_rank`).
2. Immediately afterward, shuffled indices to be used for surrogate data (described later) are generated (as specified by `surrogate_num`).
3. The `UnivariateProfiler.calc` method is executed to calculate the IPC within the specified range of degrees and time delays based on the SVD results. The calculation of the target time series (and the orthogonal polynomials required for its construction) is done lazily, meaning it is computed only when needed, and the results are cached.

Since steps 1 and 2 have already been completed in the previous cell, let’s execute the `UnivariateProfiler.calc` method in the following cell to calculate the first-order capacity $C^1$.

[zh]: #
这是该算法的简单解释：
1. 创建 `UnivariateProfiler` 类的实例时，对作为第二个参数提供的状态时间序列(本例中为 `xs`)进行归一化，然后执行 SVD 并同时测量排名(实现与 `calc_regression_and_rank` 几乎相同)。
2. 紧接着，生成用于代理数据(稍后描述)的打乱索引(如 `surrogate_num` 所指定)。
3. 执行`UnivariateProfiler.calc`方法，根据SVD结果计算指定度数和时延范围内的IPC。
目标时间序列(及其构建所需的正交多项式)的计算是延迟完成的，这意味着仅在需要时才计算，并且结果被缓存。

由于步骤 1 和步骤 2 已在前面的单元中完成，因此我们在下面的单元中执行 `UnivariateProfiler.calc` 方法来计算一阶容量 $C^1$。

In [ ]:
profiler.calc(1, 1001)

[ja]: #
このセルでは $\Tau \in \{\{0\}, \{1\}, \{2\},~\ldots,~\{1000\}\}$ すなわち 目標時系列 $z \in \{\zeta^0, \zeta^1, \zeta^2,~\ldots,~\zeta^{1000} \}$ に対してそれぞれ $\mathrm{C}[x,z]$ を計算しています (`zero_offset=False`は 1始まりを指定するオプション)。
計算結果は `profiler[key]` の形で、次数の $D$ を指定して取得できます (`key`の中身が`tuple`である点に注意)。

[en]: #
In this cell, $\Tau \in \{\{0\}, \{1\}, \{2\},~\ldots,~\{1000\}\}$, that is, for each target time series $z \in \{\zeta^0, \zeta^1, \zeta^2,~\ldots,~\zeta^{1000}\}$, $\mathrm{C}[x,z]$ is being calculated (`zero_offset=False` is an option specifying 1-based indexing).
The calculation results can be retrieved in the form of `profiler[key]` by specifying the degree $D$ (note that the content of `key` is a `tuple`).

[zh]: #
在此单元格中，正在计算 $\Tau \in \{\{0\}, \{1\}, \{2\},~\ldots,~\{1000\}\}$，即对于每个目标时间序列、$z \in \{\zeta^0, \zeta^1, \zeta^2,~\ldots,~\zeta^{1000}\}$、$\mathrm{C}[x,z]$(`zero_offset=False` 是指定从 1 开始索引的选项)。
通过指定次数$D$，可以以`profiler[key]`的形式检索计算结果(注意，`key`的内容是`tuple`)。

In [ ]:
delays, ipc, surr = profiler[(1,)]

print("delays:", *delays[:3], "...", *delays[-3:])
print("ipc:", ipc.shape)
print("surr:", surr.shape)

[ja]: #
このように`profiler`各 `key` に対して3つの情報を保持しています。
一番目の`delays` は $\Tau$ のリストです。
`ipc`は計算された $\mathrm{C}[x,z]$ が `np.array` の形で格納されておりSRが2軸目に対応します。
`surr`は同時に求められたサロゲートデータに対する $\mathrm{C}[x,z]$です。
まず`ipc`の中身を確認してみましょう。

[en]: #
In this way, `profiler` holds three pieces of information for each `key`.
The first, `delays`, is a list of $\Tau$.
`ipc` stores the calculated $\mathrm{C}[x,z]$ in the form of a `np.array`, where the SR corresponds to the second axis.
`surr` represents $\mathrm{C}[x,z]$ for the surrogate data calculated simultaneously.
First, let's check the contents of `ipc`.

[zh]: #
这样，`profiler`为每个`key`保存了3条信息。
第一个 `delays` 是 $\Tau$ 的列表。
`ipc`以`np.array`的形式存储计算出的$\mathrm{C}[x,z]$，其中SR对应于第二轴。
`surr` 代表同时计算的替代数据的 $\mathrm{C}[x,z]$。
首先我们来看看`ipc`的内容。

In [ ]:
time_step = 301

fig = Figure(figsize=(8, 6))
ax = fig[0]
im, cb = ax.plot_matrix(
    ipc[:time_step, :, 0],
    y=np.array(delays)[:time_step, 0],
    x=srs,
    aspect="auto",
    cmap="jet",
    vmin=1e-4,
    vmax=1,
    zscale="log",
    yticks_kws=dict(num_tick=4),
    xticks_kws=dict(num_tick=3),
)
ax.set_xlabel("SR", fontsize=14)
ax.set_ylabel(r"$\tau$", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=12)
cb.ax.tick_params(labelsize=12)
cb.set_label(r"$\mathrm{C}[x,\zeta^\tau]$", fontsize=14)
ax.set_title(r"$D=\{1\}$", fontsize=14)

None

[ja]: #
これは前章で学習した記憶関数に他なりません。
このグラフは色のレンジを最小値 $10^{-4}$ の対数スケールで表示しています。
$\tau$ が十分に大きいときも値が完全に0にはならず、わずかに小さい値を有し続ける様子がわかります。
これは **疑似相関** と呼ばれる現象で、有限のデータしか扱えない数値計算の制約からしばしば生じるものです。
疑似相関と思われる領域での容量$\mathrm{C}$の値は非常に小さいですが、情報処理容量の計算では膨大な種類の $\mathrm{C}$ を足し合わせる必要があるため、無視できない影響を与える場合があります (例えば $\mathrm{C}^\mathrm{tot}>r$ となってしまう)。

そこで「有意」な成分と、「有意でない」疑似相関を区別するのに使用されるのが**サロゲートデータ**です。
サロゲートデータは元の時系列データをシャッフルして生成されるデータで、元のデータの統計的性質が保持されつつも時間的な依存関係が消失しています。
今回は`surrogate_num` で指定された数だけサロゲートデータを用意し同様に容量 $\mathrm{C}$ を計測し、その最大値をしきい値として使用します。
こうして得られたしきい値を超える成分だけをランダムから区別されるものとして加味します。
この手法はRandom-shuffle法<sup>[4]</sup>として呼ばれる検定法で、もともとは文献[5]で導入されました。
1000データの場合はおおよそ有意水準 $1/1000=0.001$ の検定とみなせます。
試しにSR=1.0の成分に関してサロゲートデータを見てみましょう。

[en]: #
This is none other than the memory function learned in the previous chapter.
This graph displays the color range on a logarithmic scale with a minimum value of $10^{-4}$.
Even when $\tau$ is sufficiently large, the values do not become completely zero but remain slightly small.
This phenomenon is called **spurious correlation**, which often arises due to the constraints of numerical computations that can only handle finite data.
The values of capacity $\mathrm{C}$ in regions suspected to be spurious correlation are very small, but they can have a non-negligible impact in calculating IPC, since a large number of $\mathrm{C}$ types must be summed (e.g., $\mathrm{C}^\mathrm{tot}>r$ may occur).

**Surrogate data** is used to distinguish between "significant" components and "insignificant" spurious correlations.
Surrogate data is generated by shuffling the original time series, preserving the statistical properties while eliminating temporal dependencies.
Here, surrogate data is prepared in the number specified by `surrogate_num`, and the capacity $\mathrm{C}$ is measured in the same way, with the maximum value used as the threshold.
Only components exceeding this threshold are considered distinguishable from random noise.
This method is known as the random-shuffle method<sup>[4]</sup>, originally introduced in reference [5].
For 1000 data points, this can be regarded as a test with a significance level of $1/1000=0.001$.
Let's examine the surrogate data for the component with SR=1.0 as an example.

[zh]: #
这正是上一章学到的记忆函数。
该图以对数刻度显示颜色范围，最小值为 $10^{-4}$。
即使 $\tau$ 足够大，这些值也不会完全为零，而是保持稍小。
这种现象称为**伪相关**，这种现象通常是由于只能处理有限数据的数值计算的限制而出现的。
被怀疑是虚假相关的区域中的容量$\mathrm{C}$的值非常小，但是它们在计算IPC时可以具有不可忽略的影响，因为必须对大量的$\mathrm{C}$类型进行求和(例如，可能出现$\mathrm{C}^\mathrm{tot}>r$)。

**替代数据**用于区分“显着”成分和“不显着”虚假相关性。
代理数据是通过对原始时间序列进行改组而生成的，保留统计属性，同时消除时间依赖性。
这里，以`surrogate_num`指定的数量准备替代数据，并且以相同的方式测量容量$\mathrm{C}$，使用最大值作为阈值。
只有超过此阈值的分量才被认为可以与随机噪声区分开。
该方法被称为随机洗牌方法<sup>[4]</sup>，最初在参考文献[5]中介绍。
对于 1000 个数据点，这可以视为显着性水平为 $1/1000=0.001$ 的检验。
让我们以 SR=1.0 为例检查组件的替代数据。

In [ ]:
sr_id = 9  # 9 is the index of the SR = 1.0 in `srs`.
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
for value in surr[:, sr_id, :]:
    ax.line_y(value, color="#333333", alpha=0.5, lw=0.1)
ax.line_y(surr[:, sr_id, :].max(), color="red", lw=1)
ax.plot(np.arange(0, 1001), ipc[:, 9, 0], lw=1)
ax.set_yscale("log")
ax.set_ylim([None, 1e-2])  # Comment it out to zoom out.
ax.set_xlim([0, 1000])
ax.set_ylabel(r"$\mathrm{C}[x,\zeta^\tau]$", fontsize=14)
ax.set_xlabel(r"$\tau$", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=12)
ax.set_title(f"SR={srs[sr_id]:.2f}", fontsize=14)

None

[ja]: #
灰色の線は各サロゲートデータに対する容量 $\mathrm{C}$ を、赤色の線はそのうちの最大値を示しています。
`ipc_module`ではサロゲートデータの最大値を基準にその定数倍をしきい値として設定し有意な成分を抽出します (スケールできるようにしているのはあまりに目標時系列の数が多いため、より厳しい基準がしばしば必要だからです)。
サロゲートデータを用いてしきい値を設定し、先程のグラフをもういちど描画してみましょう。
しきい値以下の成分が白抜きになっているはずです。
同時に定数倍 `max_scale` を変化させて、しきい値の大きさでどのように変化するか確認しましょう。

[en]: #
The gray lines represent the capacity $\mathrm{C}$ for each surrogate data, and the red line shows the maximum value among them.
In `ipc_module`, the maximum value of the surrogate data is used as a reference, and a constant multiple of it is set as the threshold to extract significant components (the scaling is often needed because stricter criteria are required when the number of target time series is large).
Let's set the threshold using the surrogate data and redraw the previous graph.
The components below the threshold should appear as hollow.
At the same time, vary the constant multiplier `max_scale` to see how the graph changes with the threshold size.

[zh]: #
灰线表示每个代理数据的容量 $\mathrm{C}$，红线表示其中的最大值。
在`ipc_module`中，以代理数据的最大值为参考，并设置其恒定倍数作为阈值来提取重要成分(通常需要进行缩放，因为当目标时间序列的数量较大时，需要更严格的标准)。
让我们使用代理数据设置阈值并重新绘制之前的图表。
低于阈值的组件应显示为空心。
同时，改变常数乘数 `max_scale` 以查看图形如何随阈值大小变化。

In [ ]:
time_step = 301
max_scale = 1.0

ipc_trunc = ipc * (ipc > surr.max(axis=0, keepdims=True) * max_scale)
fig = Figure(figsize=(8, 6))
ax = fig[0]
im, cb = ax.plot_matrix(
    ipc_trunc[:time_step, :, 0],
    y=np.array(delays)[:time_step, 0],
    x=srs,
    aspect="auto",
    cmap="jet",
    vmin=1e-4,
    vmax=1,
    zscale="log",
    yticks_kws=dict(num_tick=4),
    xticks_kws=dict(num_tick=3),
)
ax.set_xlabel("SR", fontsize=14)
ax.set_ylabel(r"$\tau$", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=12)
cb.ax.tick_params(labelsize=12)
cb.set_label(r"$\mathrm{C}[x,\zeta^\tau]$", fontsize=14)
ax.set_title(r"$D=\{1\}$" + f", scale={max_scale:.2f}", fontsize=14)

None

[ja]: #
さてここまで説明のため $d=1$ について計算し詳細を確認しましたが、他の次数についても同様に計算してみましょう。
次のセルは $d=2,3,4,5$ に対する容量を計算します。
それぞれ $\tau_\mathrm{max}=300,50,30,15$ を指定しています。
例えば$D=\{1,1\}$の時は、$\Tau$の候補となる$\{\tau_1, \tau_2\}$ の組み合わせは $\{0,1\}, \{0,2\},~\ldots,~\{0,300\}, \{1,2\}, \{1,3\}, \{1,4\},~\ldots,~\{1,300\},~\ldots,~\{299,300\}$ のように$\tau_\mathrm{max}=300$ 以下の時間遅れの全ての組み合わせを網羅的に計算します。
組み合わせの数が大きく、少々計算に時間がかかるのでそのまま待ってください。
計算が完了すると計算された $D$ のリストが表示されます (`profiler.keys()` で確認できます)。

[en]: #
Now that we have calculated and examined the details for $d=1$ as an example, let’s perform similar calculations for other degrees.
The next cell calculates the capacity for $d=2,3,4,5$.
For each, $\tau_\mathrm{max}=300,50,30,15$ is specified.
For example, when $D=\{1,1\}$, the combinations of $\{\tau_1, \tau_2\}$ that are candidates for $\Tau$ are $\{0,1\}, \{0,2\},~\ldots,~\{0,300\}, \{1,2\}, \{1,3\}, \{1,4\},~\ldots,~\{1,300\},~\ldots,~\{299,300\}$.
In this way, all combinations of time delays below $\tau_\mathrm{max}=300$ are exhaustively calculated.
Since the number of combinations is large, this may take some time, so please wait patiently.
Once the calculation is complete, the list of calculated $D$ will be displayed (you can check it with `profiler.keys()`).

[zh]: #
现在我们已经以 $d=1$ 为例计算并检查了详细信息，让我们对其他度数进行类似的计算。
下一个单元格计算 $d=2,3,4,5$ 的容量。
对于每个，指定了 $\tau_\mathrm{max}=300,50,30,15$。
例如，当$D=\{1,1\}$时，作为$\Tau$的候选的$\{\tau_1, \tau_2\}$的组合是$\{0,1\}, \{0,2\},~\ldots,~\{0,300\}, \{1,2\}, \{1,3\}, \{1,4\},~\ldots,~\{1,300\},~\ldots,~\{299,300\}$。
这样，$\tau_\mathrm{max}=300$以下的所有时延组合都被穷举计算出来。
由于组合数量较多，可能需要一些时间，请耐心等待。
计算完成后，会显示计算出的$D$列表(可以通过`profiler.keys()`查看)。

In [ ]:
degrees = [2, 3, 4, 5]
taus = [300, 50, 30, 15]
for deg, tau in zip(degrees, taus, strict=True):
    profiler.calc(deg, tau + 1)

print(profiler.keys())

[ja]: #
上のセルの実行が完了したら計算結果を一度保存しておきましょう。
一般に情報処理容量の計算はとても時間がかかるので、計算結果が失われないように適宜保存するのが得策です。
[`UnivariateProfiler.save`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateViewer.save)メソッドを使うと、計算結果を`npz`形式、もしくは`pkl`形式でファイルに保存できます (圧縮性と確認の容易さの観点の観点から`npz`を推奨します)。
形式は指定されたファイルの拡張子によって決まります。
`**kwargs`に別途保存しておきたい情報を入れて保存できます。
ここではスペクトル半径 `srs` も保存しておきましょう。

[en]: #
Once the above cell completes, let's save the calculation results.
In general, calculating IPC takes considerable time, so it is advisable to save results periodically to avoid losing them.
Using the [`UnivariateProfiler.save`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateViewer.save) method, results can be saved to a file in either `npz` or `pkl` format (`npz` is recommended for better compression and easier verification).
The format is determined by the file extension.
You can include additional information to save using `**kwargs`.
Here, let's also save the spectral radii `srs`.

[zh]: #
上述单元格完成后，让我们保存计算结果。
一般来说，计算IPC需要相当长的时间，因此建议定期保存结果以避免丢失。
使用 [`UnivariateProfiler.save`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateViewer.save) 方法，结果可以保存到 `npz` 或 `pkl` 格式的文件中(建议使用 `npz` 以获得更好的压缩效果和更轻松的验证)。
格式由文件扩展名决定。
您可以使用 `**kwargs` 包含要保存的附加信息。
在这里，我们还保存谱半径 `srs`。

In [ ]:
profiler.save("./result/ipc_asym.npz", srs=srs)

[ja]: #
データの回収には[`UnivariateViewer`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateViewer)クラスを使用します。
`UnivariateViwer`は`UnivariateProfiler`の親クラスで、`UnivariateProfiler`と同じように結果を回収できますが、もとの時系列やSVDの結果は保持していないため、追加で計算できない点に注意してください (`calc`関数を呼び出せない)。
下のセルでは、`UnivariateViewer`のインスタンス`viewer`によって、`profiler`を用いた時と同様の結果が得られる点を確認してください。

[en]: #
The [`UnivariateViewer`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateViewer) class is used to retrieve data.
`UnivariateViewer` is the parent class of `UnivariateProfiler` and can retrieve results in the same way as `UnivariateProfiler`.
However, note that it does not retain the original time series or the results of the SVD, so additional calculations cannot be performed (the `calc` function cannot be called).
In the cell below, confirm that the same results can be obtained using the `UnivariateViewer` instance `viewer` as when using `profiler`.

[zh]: #
[`UnivariateViewer`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateViewer) 类用于检索数据。
`UnivariateViewer`是`UnivariateProfiler`的父类，可以像`UnivariateProfiler`一样检索结果。
但请注意，它不保留原始的时间序列或SVD的结果，因此无法执行额外的计算(无法调用`calc`函数)。
在下面的单元格中，确认使用 `UnivariateViewer` 实例 `viewer` 可以获得与使用 `profiler` 时相同的结果。

In [ ]:
viewer = UnivariateViewer("./result/ipc_asym.npz")
srs = viewer.info["srs"]  # NOTE: Keyword options are stored on `info`.
print(viewer.keys())
delays, ipc, surr = viewer[(1,)]

print("delays:", *delays[:3], "...", *delays[-3:])
print("ipc:", ipc.shape)
print("surr:", surr.shape)
print("srs:", srs)

[ja]: #
#### データの可視化と解析

[en]: #
#### Data visualization and analysis

[zh]: #
#### 数据可视化和分析

[ja]: #
$d=1$のときと異なり高次の場合は可視化は容易ではありません。
$d=2$の際は前節で扱ったように2つの時間遅れ$(\tau_1, \tau_2)$平面上のカラーマップとして $C$ を描画できましたが、高次だとより多くのグラフが必要になり大変です。
そこで `ipc_module`ではいくつかの可視化のための関数が用意されています。
次のセルで使用される [`visualize_dataframe`](https://rc-bootcamp.github.io/ipc-module/helper/#ipc_module.helper.visualize_dataframe)は総容量を棒グラフとして描画する関数です。

<details><summary> 引数の詳細</summary>

- `ax`: `Axes`
    - 描画先のAxes
- `df`: `polars.DataFrame`
    - 描画するデータフレーム
- `ranks`: `Any | None`
    - 階数のリスト
    - 与えられた場合はその値で切られる
- `xticks`: `Any | None`
    - x軸の値
- `group_by`: `str`
    - 描画する方法
        - `degree`: 次数 $d$ でグループ化
        - `component`: 次数の集合 $D$ でグループ化
        - `detail`: 次数の集合 $D$ と時間遅れの集合 $\Tau$ でグループ化 (`threhold`を指定しないと描画に時間がかかるので注意！)
- `threshold`: `float`
    - `rest` としてまとめられる成分のしきい値
- `sort_by`: `Any`
    - ソートする方法
        - `np.nanmax`: 最大値でソート
        - `np.nanmean`: 平均値でソート
        - `np.nansum`: 合計値でソート
- `cmap`: `str`
    - カラーマップ
</details>

[en]: #

Unlike the case of $d=1$, visualization for higher orders is not straightforward.
For $d=2$, as handled in the previous section, $C$ could be visualized as a color map on the $(\tau_1, \tau_2)$ plane.
However, for higher orders, more graphs are required, making it challenging.
To address this, `ipc_module` provides several functions for visualization.
The [`visualize_dataframe`](https://rc-bootcamp.github.io/ipc-module/helper/#ipc_module.helper.visualize_dataframe) function, used in the next cell, is a function that visualizes the total capacity as a bar graph (using degree/component as groups).

<details><summary> Details of the arguments</summary>

- `ax`: `Axes`
    - The Axes to draw on
- `df`: `polars.DataFrame`
    - The DataFrame to visualize
- `ranks`: `Any | None`
    - A list of ranks
    - If provided, the values are filtered accordingly
- `xticks`: `Any | None`
    - Values for the x-axis
- `group_by`: `str`
    - The grouping method for visualization
        - `degree`: Grouped by degree $d$
        - `component`: Grouped by the set of degrees $D$
        - `detail`: Grouped by the set of degrees $D$ and the set of time delays $\Tau$ (Note: visualization may take time if `threshold` is not specified!)
- `threshold`: `float`
    - Threshold for components summarized as `rest`
- `sort_by`: `Any`
    - Sorting method
        - `np.nanmax`: Sort by maximum value
        - `np.nanmean`: Sort by mean value
        - `np.nansum`: Sort by total value
- `cmap`: `str`
    - Color map
</details>

[zh]: #
与 $d=1$ 的情况不同，高阶的可视化并不简单。
对于 $d=2$，如上一节中处理的，$C$ 可以可视化为 $(\tau_1, \tau_2)$ 平面上的颜色图。
然而，对于更高的订单，需要更多的图表，这使得它具有挑战性。
为了解决这个问题，`ipc_module` 提供了多种可视化功能。
下一个单元格中使用的 [`visualize_dataframe`](https://rc-bootcamp.github.io/ipc-module/helper/#ipc_module.helper.visualize_dataframe) 函数是将总容量可视化为条形图(使用度数/分量作为组)的函数。

<details><summary> 参数详细信息</summary>

- `ax`：`Axes`
    - 绘制的轴
- `df`：`polars.DataFrame`
    - 可视化的 DataFrame
- `ranks`: `Any | None`
    - 等级列表
    - 如果提供，则相应地过滤值
- `xticks`：`Any | None`
    - x 轴的值
- `group_by`：`str`
    - 可视化的分组方法
        - `degree`：按度数分组 $d$
        - `component`：按度数 $D$ 分组
        - `detail`：按度数组 $D$ 和时间延迟组 $\Tau$ 分组(注意：如果未指定 `threshold`，可视化可能需要时间！)
- `threshold`: `float`
    - 组件阈值汇总为 `rest`
- `sort_by`: `Any`
    - 排序方法
        - `np.nanmax`：按最大值排序
        - `np.nanmean`：按平均值排序
        - `np.nansum`：按总价值排序
- `cmap`: `str`
    - 彩色地图

In [ ]:
df, rank = viewer.to_dataframe(max_scale=2.0)  # NOTE: Threshold is scaled by max_scale.
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw=dict(hspace=0.5))

for idx, group_by in enumerate(["degree", "component"]):
    ax = axes[idx]
    visualize_dataframe(
        ax,
        df,
        xticks=srs,
        threshold=0.1,
        cmap="tab10",
        group_by=group_by,  # NOTE: Either "degree" or "component" are available.
        fontsize=12,
    )
    ax.legend(
        loc="upper right",
        fontsize=12,
        bbox_to_anchor=(0.99, 0.9),
        borderaxespad=0,
        frameon=False,
    )
    ax.plot(srs, rank, ls=":", color="k")
    ax.set_xticks([0.0, 0.5, 1.0, 1.5])
    ax.set_xlabel("SR", fontsize=14)
    ax.set_ylabel(r"$\mathrm{C}$", fontsize=14)
axes[0].set_title(r"group by $d$: degree")
axes[1].set_title(r"group by $D$: the set of degree")

None

[ja]: #
上のセルで先に登場しましたが[`UnivariateViewer.to_dataframe`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateViewer.to_dataframe)メソッドを使うと、計算結果を[`polars.DataFrame`](https://docs.pola.rs/py-polars/html/reference/dataframe/)形式で計算結果を取得できます。
[`polars`](https://pola.rs/)は[`pandas`](https://pandas.pydata.org/)と同じデータ解析のためのライブラリですが、より高速に動作します。

[en]: #
As mentioned earlier in the above cell, the [`UnivariateViewer.to_dataframe`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateViewer.to_dataframe) method can be used to retrieve the calculation results in the [`polars.DataFrame`](https://docs.pola.rs/py-polars/html/reference/dataframe/) format.
[`polars`](https://pola.rs/) is a data analysis library similar to [`pandas`](https://pandas.pydata.org/), but it operates much faster.

[zh]: #
如上单元格中前面提到的，[`UnivariateViewer.to_dataframe`](https://rc-bootcamp.github.io/ipc-module/profiler/#ipc_module.profiler.UnivariateViewer.to_dataframe) 方法可用于检索 [`polars.DataFrame`](https://docs.pola.rs/py-polars/html/reference/dataframe/) 格式的计算结果。
[`polars`](https://pola.rs/)是一个与[`pandas`](https://pandas.pydata.org/)类似的数据分析库，但运行速度要快得多。

In [ ]:
df, rank = viewer.to_dataframe(max_scale=2.0)  # NOTE: Threshold is scaled by max_scale.
df

[ja]: #
`polars`の詳細な使い方は[公式のドキュメント](https://docs.pola.rs/user-guide/getting-started/)を参照してください。
次の節からは代表的な描画方法を紹介します。

[en]: #
Refer to the [official documentation](https://docs.pola.rs/user-guide/getting-started/) for detailed usage of `polars`.
The next section introduces representative visualization methods.

[zh]: #
`polars`的详细使用方法请参考 [官方文档](https://docs.pola.rs/user-guide/getting-started/)。
下一节介绍代表性的可视化方法。

[ja]: #
##### $|D|=1$ の描画

[en]: #
##### Visualization for $|D|=1$

[zh]: #
##### $|D|=1$ 的可视化

[ja]: #
$D$ の要素数が1、すなわち$D\in\{\{1\}, \{2\}, \{3\},~\ldots\} $ の場合はそのまま容量 $\mathrm{C}$ を一次元のグラフとして描画できます。
`viewer.to_dataframe`の引数に負の値を指定すると、$D$ の要素数がその絶対値のものだけを抽出できます (例 `df = viewer.to_dataframe(-1)`)。
以下のグラフでは各スペクトル半径のデータに対して、$\mathrm{C}[x, \mathcal{P}_d(\zeta^\tau)]$ を描画します。
また`DataFrame`内の要素を足し合わせる以外にも、`viewer.total`メソッドを用いて総容量を計算できます。

[en]: #
If the number of elements in $D$ is 1, i.e., $D \in \{\{1\}, \{2\}, \{3\},~\ldots\}$, the capacity $\mathrm{C}$ can be visualized as a one-dimensional graph.
By specifying a negative value as an argument to `viewer.to_dataframe`, you can extract only those with the absolute value of the number of elements in $D$ (e.g., `df = viewer.to_dataframe(-1)`).
In the graph below, $\mathrm{C}[x, \mathcal{P}_d(\zeta^\tau)]$ is visualized for the data of each spectral radius.
Additionally, instead of summing the elements in the `DataFrame`, you can use the `viewer.total` method to calculate the total capacity.

[zh]: #
如果$D$中的元素数量为1，即$D \in \{\{1\}, \{2\}, \{3\},~\ldots\}$，则容量$\mathrm{C}$可以可视化为一维图。
通过指定负值作为 `viewer.to_dataframe` 的参数，您可以仅提取那些具有 $D$ 中元素数量绝对值的元素(例如，`df = viewer.to_dataframe(-1)`)。
在下图中，$\mathrm{C}[x, \mathcal{P}_d(\zeta^\tau)]$ 对每个谱半径的数据进行了可视化。
此外，您可以使用 `viewer.total` 方法来计算总容量，而不是对 `DataFrame` 中的元素求和。

In [ ]:
max_scale = 2.0
degrees = [1, 2, 3]
sr_ids = [4, 9, 14]

df, rank = viewer.to_dataframe(-1, max_scale=0.0)  # NOTE: No truncation.

grid_size = (len(sr_ids), len(degrees))
fig, axes = plt.subplots(
    *grid_size, figsize=(grid_size[1] * 4, grid_size[0] * 3), gridspec_kw=dict(hspace=0.1, wspace=0.1)
)

for (idy, sr_id), (idx, degree) in itertools.product(enumerate(sr_ids), enumerate(degrees)):
    scale = viewer.calc_surr_max((degree,), max_scale=max_scale)[sr_id, 0]
    capacity = viewer.total((degree,), max_scale=max_scale)[sr_id]
    ax = axes[idy, idx]
    df_sub = df.filter(df["degree"] == degree).sort("del_0")  # NOTE: Filter by degree.
    delays = df_sub["del_0"]
    ipc = df_sub[f"ipc_{sr_id}"]
    ax.plot(
        delays,
        ipc,
        color=f"C{idx}",
        lw=1,
        label=f"SR={srs[sr_id]:.2f}",
    )
    ax.line_y(scale, color="red", lw=1, ls="--")
    ax.set_yscale("log")
    ax.set_ylim([1e-4, 1.1])
    ax.tick_params(axis="both", which="major", labelsize=12)
    if idy == 0:
        ax.set_title(f"$d={degree}$", fontsize=14)
    if idx == 0:
        ax.set_ylabel("SR=" + f"{srs[sr_id]:.2f}", fontsize=14)
    else:
        ax.set_yticklabels([])
    if idy < len(sr_ids) - 1:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel(r"$\tau$", fontsize=14)
    ax.text(
        0.95,
        0.95,
        "C={:.2f}".format(capacity),
        fontsize=12,
        ha="right",
        va="top",
        transform=ax.transAxes,
    )
fig.suptitle(r"$\mathrm{C}[x,\mathcal{P}_d(\zeta^\tau)]$" + f", scale={max_scale:.2f}", fontsize=16)
None

[ja]: #
##### $D=\{1,1\}$ の描画

[en]: #
##### Visualization for $D=\{1,1\}$

[zh]: #
##### $D=\{1,1\}$ 的可视化

[ja]: #
今度は$D=\{1,1\}$ の場合を考えます。
$|D|=1$ の時とは異なり、$D$ の要素数が2つあるため、2次元の平面上に描画できます。
`df = viewer.to_dataframe((1, 1))`とすると、$D=\{1,1\}$ の成分のみを抽出した`DataFrame`を取得できます。

[en]: #
Now, let us consider the case where $D=\{1,1\}$.
Unlike when $|D|=1$, since $D$ has two elements, it can be visualized on a two-dimensional plane.
By using `df = viewer.to_dataframe((1, 1))`, you can obtain a `DataFrame` that extracts only the components for $D=\{1,1\}$.

[zh]: #
现在，让我们考虑 $D=\{1,1\}$ 的情况。
与 $|D|=1$ 不同，由于 $D$ 有两个元素，因此可以在二维平面上可视化。
通过使用 `df = viewer.to_dataframe((1, 1))`，您可以获得仅提取 $D=\{1,1\}$ 组件的 `DataFrame`。

In [ ]:
max_scale = 2.0

df, rank = viewer.to_dataframe((1, 1), max_scale=max_scale)
capacities = viewer.total((1, 1), max_scale=max_scale)
grid_size = (4, 5)
fig = Figure(figsize=(grid_size[1] * 3, grid_size[0] * 3))
fig.create_grid(*grid_size, hspace=0.3, wspace=0.3)

pos = len(srs)
ax_last = fig[pos // grid_size[1], pos % grid_size[1]]
ax_last.create_grid(1, 2, width_ratios=[1, 20])
cax = ax_last[0]
cax.tick_params(labelsize=12)
for idx, sr in enumerate(srs):
    ax = fig[idx // grid_size[1], idx % grid_size[1]]
    df_pivot = df.sort("del_0", "del_1").pivot(
        "del_1",
        index="del_0",
        values=f"ipc_{idx}",
    )
    mat = df_pivot[:, 1:]
    index = df_pivot[:, 0]
    columns = list(map(int, df_pivot.columns[1:]))
    ax.plot_matrix(
        mat,
        index=index,
        column=columns,
        cmap="viridis",
        zscale="log",
        vmin=1e-4,
        vmax=1,
        aspect="equal",
        xticks_kws=dict(num_tick=7),
        yticks_kws=dict(num_tick=7),
        colorbar=(idx == len(srs) - 1),
        cax=cax if (idx == len(srs) - 1) else None,
    )
    ax.set_xlim(-0.5, 50.5)
    ax.set_ylim(-0.5, 50.5)
    ax.tick_params(axis="both", which="major", labelsize=12)
    ax.text(
        0.95,
        0.95,
        "C={:.2f}".format(capacities[idx]),
        fontsize=12,
        ha="right",
        va="top",
        transform=ax.transAxes,
    )
    ax.set_title(f"SR={sr:.2f}", fontsize=14)

for idx in range(len(srs), grid_size[0] * grid_size[1]):
    fig.delaxes(fig[idx // grid_size[1], idx % grid_size[1]])

[ja]: #
#### 応用例: 入力の対称性の影響の確認

[en]: #
#### Application example: confirming the effect of input symmetry

[zh]: #
#### 应用示例：确认输入对称的效果

[ja]: #
最後に情報処理容量の有効性を示す例として、対称入力に対する影響を確認しましょう。
先ほどと全く同じ条件のESNですが、入力のスケールを $[-1, 1]$ のままにしてESNに与えてみます。

[en]: #
Finally, as an example to demonstrate the effectiveness of IPC, let’s examine the impact on symmetric input.
Using the same ESN conditions as before, provide the input to the ESN while keeping the scale at $[-1, 1]$.

[zh]: #
最后，作为展示 IPC 有效性的示例，我们来看看对对称输入的影响。
使用与之前相同的 ESN 条件，向 ESN 提供输入，同时将规模保持在 $[-1, 1]$。

In [ ]:
seed = 5678
dim = 50
t_washout = 10000
t_sample = 100000
srs = np.linspace(0.1, 1.7, 17)
t_total = t_washout + t_sample
display = True

rnd = np.random.default_rng(seed)
w_in = Linear(1, dim, bound=0.1, bias=0.0, rnd=rnd)
net = ESN(dim, sr=srs[:, None], f=np.tanh, p=1, rnd=rnd)

x0 = np.zeros((srs.shape[0], dim))
us = rnd.uniform(-1, 1, (t_total, 1))

x = x0
xs = np.zeros((t_total, *x0.shape))
for idx in trange(t_total, display=display):
    x = net(x, w_in(us[idx]))  # NOTE: Use us[idx] for the symmetric case
    xs[idx] = x

print("us:", us.shape)
print("xs:", xs.shape)

[ja]: #
先ほどと全く同じ条件で情報処理容量の計測を行います。
同じく時間がかかるのでしばらくお待ちください。
結果を`./output/ipc_symm.npz`として保存します。

[en]: #
We will measure the IPC under the same conditions as before.
This will also take some time, so please wait.
The results will be saved as `./output/ipc_symm.npz`.

[zh]: #
我们将在与之前相同的条件下测量 IPC。
这也需要一些时间，所以请等待。
结果将保存为 `./output/ipc_symm.npz`。

In [ ]:
use_gpu = True  # NOTE: Set it to False to run on CPU.

if use_gpu:
    import torch

    assert torch.cuda.is_available(), "CUDA is not available"
    us_c = torch.from_numpy(us).cuda()
    xs_c = torch.from_numpy(xs).cuda()
    args = (us_c, xs_c)
else:
    args = (us, xs)

profiler = UnivariateProfiler(
    *args,
    "Legendre",
    offset=t_washout,
    surrogate_num=1000,
    axis1=0,
    axis2=-1,
)

degrees = [1, 2, 3, 4, 5]
taus = [1000, 300, 50, 30, 15]
for deg, tau in zip(degrees, taus, strict=True):
    profiler.calc(deg, tau + 1)

print(profiler.keys())

profiler.save("./result/ipc_symm.npz", srs=srs)

[ja]: #
以下のセルは`ipc_asym.npz`と`ipc_symm.npz`両方のデータを読み込み、図表として比較します。
後から計算された`ipc_symm.npz`の方は対称入力で活性化関数$\tanh$ は奇関数であるため、偶数次数の成分の消失が期待されます。
果たしてどうなるでしょうか？

[en]: #
The following cell loads data from both `ipc_asym.npz` and `ipc_symm.npz` and compares them in a figure.
For `ipc_symm.npz`, which uses symmetric input, the even-order components are expected to vanish because the activation function $\tanh$ is an odd function.
What will the result be?

[zh]: #
以下单元格加载 `ipc_asym.npz` 和 `ipc_symm.npz` 的数据，并在图中对它们进行比较。
对于使用对称输入的 `ipc_symm.npz`，偶数阶分量预计会消失，因为激活函数 $\tanh$ 是奇函数。
结果会怎样呢？

In [ ]:
files = ["./result/ipc_asym.npz", "./result/ipc_symm.npz"]

fig, axes = plt.subplots(1, len(files), figsize=(16, 6), gridspec_kw=dict(hspace=0.5))
for idx, file in enumerate(files):
    viewer = UnivariateViewer(file)
    srs = viewer.info["srs"]
    df, rank = viewer.to_dataframe(max_scale=2.0)  # NOTE: Threshold is scaled by max_scale.
    ax = axes[idx]
    visualize_dataframe(
        ax,
        df,
        xticks=srs,
        threshold=0.1,
        cmap="tab10",
        group_by="component",
        fontsize=12,
    )
    ax.legend(
        loc="upper right",
        fontsize=12,
        bbox_to_anchor=(0.99, 0.9),
        borderaxespad=0,
        frameon=False,
    )
    ax.plot(srs, rank, ls=":", color="k")
    ax.set_xticks([0.0, 0.5, 1.0, 1.5])
    ax.set_xlabel("SR", fontsize=14)
    ax.set_ylabel(r"$\mathrm{C}$", fontsize=14)
    ax.set_title(file, fontsize=16)

None

[ja]: #
前章同様に分岐図と条件付きLyapunov指数の計算と合わせて描画して比較してみましょう。

[en]: #
As in the previous chapter, let’s plot and compare the bifurcation diagram along with the calculation of the conditional Lyapunov exponent.

[zh]: #
与上一章一样，我们绘制并比较分岔图以及条件李雅普诺夫指数的计算。

In [ ]:
eps = 1e-4
net.sr = np.linspace(0.1, 1.7, 161)[:, None]

x0 = np.zeros((2, net.sr.shape[0], dim))
us = rnd.uniform(-1, 1, (t_total, 1))

t_washout, t_sample = 1000, 20000
ts = np.arange(-t_washout, t_sample)

x = x0
xs = np.zeros((t_total, *x0.shape[1:]))
lmbds = np.zeros((t_sample, net.sr.shape[0]))
for idx, t in enumerate(tqdm(ts, display=display)):
    if t == 0:
        pert = rnd.uniform(-1, 1, x[0].shape)
        pert = pert / np.linalg.norm(pert, axis=-1, keepdims=True)
        x[1] = x[0] + pert * eps
    x = net(x, w_in(us[idx]))
    xs[idx] = x[0]
    if t >= 0:
        x_org, x_per = x[0], x[1]
        x_diff = x_per - x_org
        d_post = np.linalg.norm(x_diff, axis=-1, keepdims=True)
        lmbd = np.log(np.abs(d_post / eps))
        x_per[:] = x_org + x_diff * (eps / d_post)
        lmbds[idx - t_washout] = lmbd[..., 0]


def get_maxima_and_minima(xs, **kwargs):
    id_maxima = sp.signal.find_peaks(xs, **kwargs)[0]
    id_minima = sp.signal.find_peaks(-xs, **kwargs)[0]
    return id_maxima, id_minima


fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 10), gridspec_kw=dict(hspace=0.05))
axl = axes[0]
axl.set_xticklabels([])
for idx, sr in enumerate(net.sr):
    id_maxima, id_minima = get_maxima_and_minima(xs[t_washout:, idx, 0])
    id_all = np.concatenate([id_maxima, id_minima])
    peaks = xs[t_washout:, idx, 0][id_all]
    axl.scatter(sr * np.ones(peaks.shape[0]), peaks, marker=".", s=0.01, color="k")
axl.tick_params(axis="both", which="major", labelsize=12)
axl.set_ylabel(r"$x_0[k]$", fontsize=14)
axl.set_yticks([-1.0, 0.0, 1.0])
axl.set_ylim(-1.1, 1.1)

axr = axes[0].twinx()
axr.plot(net.sr, lmbds.mean(axis=0), "o-", color="red", label="MLE")
axr.set_yticks([-0.2, 0.0, 0.2])
axr.set_ylim(-0.22, 0.22)
axr.set_ylabel(r"MLE: $\lambda$", fontsize=14)
axr.set_xticklabels([])
axr.tick_params(axis="both", which="major", labelsize=12)

viewer = UnivariateViewer("./result/ipc_symm.npz")
srs = viewer.info["srs"]
df, rank = viewer.to_dataframe(max_scale=2.0)  # NOTE: Threshold is scaled by max_scale.
ax = axes[1]
visualize_dataframe(
    ax,
    df,
    xticks=srs,
    threshold=0.1,
    cmap="tab10",
    group_by="component",
    fontsize=12,
)
ax.legend(
    loc="upper right",
    fontsize=12,
    bbox_to_anchor=(0.99, 0.9),
    borderaxespad=0,
    frameon=False,
)
ax.set_xlabel("SR", fontsize=14)
ax.set_ylabel(r"$\mathrm{C}$", fontsize=14)

for ax in [axl, axr, axes[1]]:
    ax.plot(srs, rank, ls=":", color="k")
    ax.set_xticks([0.0, 0.5, 1.0, 1.5, 2.0])
    ax.set_xlim(srs.min() - 0.1, srs.max() + 0.1)
fig.align_labels()

[ja]: #
Q3.1. (Advanced)
- `narma_func`を用い、NARMA10に対して情報処理容量を計測し、[5]のFigure 3の結果を再現せよ。
- NARMAとESNの情報処理容量を比較し、NARMAが解けるESNの条件を考察せよ。

[en]: #
- Use `narma_func` to measure the IPC for NARMA10 and reproduce the results in Figure 3 of [5].
- Compare the IPC of NARMA and ESN, and discuss the conditions under which the ESN can solve NARMA.

[zh]: #
- 使用 `narma_func` 测量 NARMA10 的 IPC 并重现 [5] 的图 3 中的结果。
- 比较NARMA和ESN的IPC，并讨论ESN在什么条件下可以解决NARMA。

[ja]: #
## 参考文献

[en]: #
## References

[zh]: #
## 参考文献

[1] Dambre, J., Verstraeten, D., Schrauwen, B., & Massar, S. (2012). *Information Processing Capacity of Dynamical Systems*. Scientific Reports, 2(1), 514. https://doi.org/10.1038/srep00514

[2] Xiu, D., & Karniadakis, G. E. (2002). *The Wiener--Askey Polynomial Chaos for Stochastic Differential Equations*. SIAM Journal on Scientific Computing, 24(2), 619–644. https://doi.org/10.1137/S1064827501387826

[3] Hardy, G. H., & Ramanujan, S. (1918). *Asymptotic Formulaæ in Combinatory Analysis*. Proceedings of the London Mathematical Society, s2-17(1), 75–115. https://doi.org/10.1112/plms/s2-17.1.75

[4] Theiler, J., Eubank, S., Longtin, A., Galdrikian, B., & Doyne Farmer, J. (1992). *Testing for Nonlinearity in Time Series: The Method of Surrogate Data*. Physica D: Nonlinear Phenomena, 58(1), 77–94. https://doi.org/10.1016/0167-2789(92)90102-S

[5] Kubota, T., Takahashi, H., & Nakajima, K. (2021). *Unifying Framework for Information Processing in Stochastically Driven Dynamical Systems*. Physical Review Research, 3(4), 043135. https://doi.org/10.1103/PhysRevResearch.3.043135